# Forklift Target Anomaly Inference

保存済みの特徴量ベース異常検知モデルを読み込み、複数の前処理済み動画に対して窓別スコアを推論します。学習処理は含めません。


学習ノートで保存した artifact を読み込み、同じ特徴量定義・同じ前処理 pipeline でターゲット動画を採点します。`random_untrained` を使うと、学習に使っていない前処理済み動画から比率指定で対象を選べます。


## 1. セットアップ

In [1]:
# ------------------------------------------------------------
# セル概要: 推論セットアップ
# ------------------------------------------------------------
# - 学習済み artifact を読み込み、推論時の前処理条件を学習時 settings で上書きします。
# - RUN_INFERENCE / PLOT_* のフラグで、スコア計算や可視化の重さを調整できます。
# ------------------------------------------------------------

# ============================================================
# ライブラリ読み込み
# ------------------------------------------------------------
# このノートブックで使う標準ライブラリ、画像処理、音声処理、
# 機械学習、描画系のライブラリをまとめて読み込みます。
# tqdm / librosa は環境によって入っていない可能性があるため、
# import に失敗しても最低限処理が続くようフォールバックしています。
# ============================================================
from __future__ import annotations

import math
import warnings
from pathlib import Path

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import butter, filtfilt, resample_poly
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

try:
    import librosa
except Exception as exc:
    librosa = None
    warnings.warn(f"librosa could not be imported. Audio features will use a scipy fallback. reason={exc}")

plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_columns", 120)


# 関数メモ: find_project_root
# - 現在の作業ディレクトリから親方向へたどり、リポジトリの基準ディレクトリを見つけます。
# - Notebook を `notebooks/` から実行しても repo root から実行しても、入出力パスがずれないようにします。

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "movie_preprocess").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repository root from {start}.")


# 推論時も repo root を見つけ、学習済みモデルと出力先を同じ場所にそろえます。
PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "anomaly_feature_poc"
MODEL_PATH = OUTPUT_DIR / "isolation_forest_feature_poc.joblib"

# RUN_INFERENCE=True のときは学習済みモデルを使って異常スコアまで計算します。
# False にすると特徴量と raw flow の確認だけを行えます。
RUN_INFERENCE = True
# スコアグラフなど、推論結果の既存グラフを出すかどうかを切り替えます。
PLOT_EXISTING_GRAPHS = False
# 特徴量グラフを出すかどうかを切り替えます。False の場合は raw flow / 広域振動 / 特徴量 overlay を保存しません。
PLOT_FEATURE_GRAPHS = False
PLOT_RAW_FLOW_XY = True
RAW_FLOW_SAMPLE_SEC = 0.1
RAW_FLOW_MAGNITUDE_LOWPASS_CUTOFF_HZ = 1.0
RAW_FLOW_MAGNITUDE_LOWPASS_ORDER = 2
FLOW_SCORE_WINDOW_SEC = 1.0
FLOW_SCORE_HOP_SEC = 0.5
FLOW_SCORE_ALPHA_MIN = 0.04
FLOW_SCORE_ALPHA_MAX = 0.42
FLOW_SCORE_MIN_VISIBLE = 1e-6
FLOW_SCORE_HIGH_RATIO_FRACTION = 0.5
FLOW_SCORE_DISTINCTIVENESS_WEIGHT = 0.85

# 学習ノートで保存した artifact から、モデル本体・前処理・設定値を読み込みます。
if MODEL_PATH.exists():
    artifacts = joblib.load(MODEL_PATH)
else:
    if RUN_INFERENCE:
        raise FileNotFoundError(f"Trained model was not found: {MODEL_PATH}")
    warnings.warn(f"Trained model was not found. Raw flow extraction will continue without inference: {MODEL_PATH}")
    artifacts = {}

# settings には学習時の窓幅、flow グリッド、閾値などが入っています。
# ここで読み戻すことで、学習時と推論時の特徴量定義を一致させます。
settings = artifacts.get("settings", {})
FLOW_SCORE_WINDOW_SEC = float(settings.get("flow_score_window_sec", FLOW_SCORE_WINDOW_SEC))
FLOW_SCORE_HOP_SEC = float(settings.get("flow_score_hop_sec", FLOW_SCORE_HOP_SEC))
FLOW_SCORE_ALPHA_MIN = float(settings.get("flow_score_alpha_min", FLOW_SCORE_ALPHA_MIN))
FLOW_SCORE_ALPHA_MAX = float(settings.get("flow_score_alpha_max", FLOW_SCORE_ALPHA_MAX))
FLOW_SCORE_MIN_VISIBLE = float(settings.get("flow_score_min_visible", FLOW_SCORE_MIN_VISIBLE))
FLOW_SCORE_HIGH_RATIO_FRACTION = float(settings.get("flow_score_high_ratio_fraction", FLOW_SCORE_HIGH_RATIO_FRACTION))
FLOW_DIRECTION_MIN_MAG = float(settings.get("flow_direction_min_mag", 0.025))
DIRECTION_JITTER_HIGH_THRESHOLD = float(settings.get("direction_jitter_high_threshold", 1.5))
DIRECTION_JITTER_LOW_THRESHOLD = float(settings.get("direction_jitter_low_threshold", 0.8))
DIRECTION_JITTER_THRESHOLD_MODE = str(settings.get("direction_jitter_threshold_mode", "percentile")).lower()
DIRECTION_JITTER_HIGH_PERCENTILE = float(settings.get("direction_jitter_high_percentile", 85.0))
DIRECTION_JITTER_LOW_PERCENTILE = float(settings.get("direction_jitter_low_percentile", 60.0))
DIRECTION_JITTER_LOW_EXPANSION_STEPS = int(settings.get("direction_jitter_low_expansion_steps", 2))
DIRECTION_JITTER_SCORE_LOWER_PERCENTILE = float(settings.get("direction_jitter_score_lower_percentile", 0.0))
DIRECTION_JITTER_SCORE_UPPER_PERCENTILE = float(settings.get("direction_jitter_score_upper_percentile", 95.0))

# audio / motion の個別モデルを取り出します。古い artifact しかない場合は互換用に combined として扱います。
score_model_artifacts = artifacts.get("score_models")
if not score_model_artifacts and artifacts:
    legacy_feature_names = np.asarray(artifacts["feature_names"])
    score_model_artifacts = {
        "combined": {
            "model_name": "combined",
            "model": artifacts["model"],
            "preprocess_pipeline": artifacts["preprocess_pipeline"],
            "feature_names": legacy_feature_names,
            "feature_weight_vector": np.asarray(artifacts.get("feature_weight_vector", np.ones(len(legacy_feature_names))), dtype=float),
            "score_calibration": {"lower": 0.0, "upper": 1.0},
            "score_column": "anomaly_score",
            "raw_score_column": "anomaly_score_raw",
            "expanded_feature_group_map": artifacts.get("expanded_feature_group_map", {}),
        }
    }
score_model_artifacts = score_model_artifacts or {}
score_model_configs = artifacts.get("score_model_configs", {})
sync_score_config = artifacts.get("sync_score_config", {"audio_score_column": "audio_anomaly_score", "motion_score_column": "motion_anomaly_score", "max_lag_windows": 2})
final_score_weights = artifacts.get("final_score_weights", {"audio_anomaly_score": 0.45, "motion_anomaly_score": 0.35, "sync_score": 0.20})

# グラフへ出す主要スコア列です。特徴量側は、解釈しやすい音声代表値と広域振動スコアに絞っています。
PLOT_SCORE_COLUMNS = ["anomaly_score", "audio_anomaly_score", "motion_anomaly_score", "sync_score"]
PLOT_FEATURE_COLUMNS = ["audio_rms", "audio_high_freq_energy", "front_broad_vibration_score", "rear_broad_vibration_score"]

# ここから下は artifact の settings を優先して、学習時と同じ前処理条件を再現します。
FPS_SAMPLE = int(settings.get("fps_sample", 10))
WINDOW_SEC = float(settings.get("window_sec", 0.2))
AUDIO_SR = int(settings.get("audio_sr", 16000))
N_MELS = int(settings.get("n_mels", 16))
FRAME_RESIZE_WIDTH = int(settings.get("frame_resize_width", 480))
FLOW_ANALYSIS_SCALE = float(settings.get("flow_analysis_scale", 0.75))
FLOW_GAUSSIAN_BLUR_KERNEL = settings.get("flow_gaussian_blur_kernel", (5, 5))
FLOW_GAUSSIAN_BLUR_SIGMA = float(settings.get("flow_gaussian_blur_sigma", 0.0))
FLOW_GRID = tuple(settings.get("flow_grid", (3, 3)))
FLOW_MIN_VALID_CELL_RATIO = float(settings.get("flow_min_valid_cell_ratio", 0.01))
TOP_N_ANOMALIES = 8
BROAD_VIBRATION_TOP_K = int(settings.get("broad_vibration_top_k", artifacts.get("broad_vibration_top_k", 5)))
BROAD_VIBRATION_COLUMNS = list(artifacts.get("broad_vibration_columns", settings.get("broad_vibration_columns", ["front_broad_vibration_score", "rear_broad_vibration_score"])))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"MODEL_PATH  : {MODEL_PATH}")
print(f"model_version: {artifacts.get('model_version', 'legacy')}")
print(f"RUN_INFERENCE: {RUN_INFERENCE}")
print(f"PLOT_EXISTING_GRAPHS: {PLOT_EXISTING_GRAPHS}")
print(f"PLOT_FEATURE_GRAPHS : {PLOT_FEATURE_GRAPHS}")
print(f"PLOT_RAW_FLOW_XY: {PLOT_RAW_FLOW_XY}")
print(f"FLOW_SCORE_WINDOW_SEC: {FLOW_SCORE_WINDOW_SEC}")
print(f"FLOW_SCORE_HOP_SEC: {FLOW_SCORE_HOP_SEC}")
print(f"DIRECTION_JITTER_THRESHOLD_MODE: {DIRECTION_JITTER_THRESHOLD_MODE}")
print(f"DIRECTION_JITTER_HIGH_PERCENTILE: {DIRECTION_JITTER_HIGH_PERCENTILE}")
print(f"DIRECTION_JITTER_LOW_PERCENTILE : {DIRECTION_JITTER_LOW_PERCENTILE}")
print(f"BROAD_VIBRATION_TOP_K: {BROAD_VIBRATION_TOP_K}")
print(f"BROAD_VIBRATION_COLUMNS: {BROAD_VIBRATION_COLUMNS}")
print(f"score models: {list(score_model_artifacts.keys())}")
for model_name, artifact in score_model_artifacts.items():
    print(f"- {model_name}: feature_dim={len(artifact['feature_names'])}, score_column={artifact.get('score_column')}")
print(f"sync score config : {sync_score_config}")
print(f"final score weights: {final_score_weights}")
print(f"plot score columns : {PLOT_SCORE_COLUMNS}")
print(f"plot feature columns: {PLOT_FEATURE_COLUMNS}")


PROJECT_ROOT: /workspaces/forklift
MODEL_PATH  : /workspaces/forklift/outputs/anomaly_feature_poc/isolation_forest_feature_poc.joblib
model_version: audio_motion_sync_v10_broad_vibration_clean
RUN_INFERENCE: True
PLOT_EXISTING_GRAPHS: False
PLOT_FEATURE_GRAPHS : False
PLOT_RAW_FLOW_XY: True
FLOW_SCORE_WINDOW_SEC: 1.0
FLOW_SCORE_HOP_SEC: 0.5
DIRECTION_JITTER_THRESHOLD_MODE: percentile
DIRECTION_JITTER_HIGH_PERCENTILE: 85.0
DIRECTION_JITTER_LOW_PERCENTILE : 60.0
BROAD_VIBRATION_TOP_K: 5
BROAD_VIBRATION_COLUMNS: ['front_broad_vibration_score', 'rear_broad_vibration_score']
score models: ['audio', 'motion']
- audio: feature_dim=25, score_column=audio_anomaly_score
- motion: feature_dim=2, score_column=motion_anomaly_score
sync score config : {'audio_score_column': 'audio_anomaly_score', 'motion_score_column': 'motion_anomaly_score', 'max_lag_windows': 2}
final score weights: {'audio_anomaly_score': 0.45, 'motion_anomaly_score': 0.35, 'sync_score': 0.2}
plot score columns : ['anomaly_score'

## 2. 推論対象

`TARGET_SELECTION_MODE` で推論対象の選び方を切り替えます。

- `manual`: `TARGET_SPECS` に書いた `sample_id` を推論します。`category` / `environment` は `movie_preprocess` から自動判定します。
- `random_untrained`: 学習に使っていない前処理済みデータから、件数と normal/anomaly・indoor/outdoor 比率に従ってランダム抽出します。`RANDOM_TARGET_COUNT` が負数なら未学習データ全件を使います。


In [2]:
# ------------------------------------------------------------
# セル概要: 推論対象の選択
# ------------------------------------------------------------
# - manual または random_untrained で推論対象を決め、front/rear/audio の実パスを解決します。
# - 学習済み sample_id を除外できるよう train_sample_list.csv も参照します。
# ------------------------------------------------------------

# ============================================================
# 推論対象サンプルの定義
# ------------------------------------------------------------
# 推論対象は2つのモードで選べます。
# manual          : TARGET_SPECS に sample_id を直接書いた動画を処理します。
# random_untrained: 学習に使っていない movie_preprocess 済み動画からランダム抽出します。
# category / environment は movie_preprocess の配置・manifest から自動で判定します。
# ============================================================
# TARGET_SELECTION_MODE = "manual"  # "manual" or "random_untrained"
TARGET_SELECTION_MODE = "random_untrained"

# manual モード用。category / environment は不要です。
# 同じ sample_id が複数カテゴリに存在する場合だけ、必要に応じて category / environment を追加してください。
TARGET_SPECS = [
    {"sample_id": "001_後進時にトラックに衝突"},
    {"sample_id": "005_後進旋回時トラックに衝突"},
    {"sample_id": "006_安全ポールに接触"},
    {"sample_id": "012_安全ポール乗り上げ"},
    {"sample_id": "090_前進時、電源ケーブル乗り上げ"},
    {"sample_id": "1001"},
    {"sample_id": "1002"},
    {"sample_id": "1006"},
    {"sample_id": "1024"},
    {"sample_id": "1036"},
]

# random_untrained モード用。
# RANDOM_TARGET_COUNT < 0 の場合は、学習に使っていない usable なデータを全件使います。
RANDOM_TARGET_COUNT = 50
RANDOM_TARGET_CATEGORY_RATIOS = {"normal": 0.5, "anomaly": 0.5}
RANDOM_TARGET_ENVIRONMENT_RATIOS = {"indoor": 0.5, "outdoor": 0.5}
RANDOM_TARGET_RANDOM_STATE = 42
TARGET_PRINT_LIMIT = 30

MOVIE_PREPROCESS_MANIFEST_PATH = PROJECT_ROOT / "data" / "movie_preprocess" / "movie_preprocess_manifest.csv"
TRAIN_SAMPLE_LIST_PATH = PROJECT_ROOT / "data" / "train_sample_list.csv"
TARGET_OUTPUT_ROOT = OUTPUT_DIR / "target_inference"
TARGET_COMPARISON_DIR = TARGET_OUTPUT_ROOT / "_comparison"
TARGET_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TARGET_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)


# 関数メモ: infer_sample_id_from_preprocess_path
# - manifest の出力パスから sample_id を復元します。
# - front/rear/audio で共通する名前を取り出し、推論対象のファイルパス構築に使います。

def infer_sample_id_from_preprocess_path(path_value, suffix: str = "_front.mp4") -> str:
    # manifest の出力パスから、front/rear/audio に共通する sample_id を取り出します。
    name = Path(str(path_value)).name
    if name.endswith(suffix):
        return name[: -len(suffix)]
    return Path(name).stem


# 関数メモ: build_preprocess_paths
# - category/environment/sample_id から前処理済み front/rear/audio のパスを組み立てます。
# - manifest の相対パスに依存せず、現在の PROJECT_ROOT 基準で安定して解決します。

def build_preprocess_paths(sample_id: str, category: str, environment: str) -> dict[str, Path]:
    # 実ファイルは data/movie_preprocess/{category}/{environment}/ にまとまっています。
    # manifest のパスが相対表記でも、この組み立てなら notebook の実行場所に依存しません。
    target_dir = PROJECT_ROOT / "data" / "movie_preprocess" / str(category) / str(environment)
    return {
        "front_path": target_dir / f"{sample_id}_front.mp4",
        "rear_path": target_dir / f"{sample_id}_rear.mp4",
        "audio_path": target_dir / f"{sample_id}_audio.wav",
    }


# 関数メモ: scan_movie_preprocess_inventory
# - manifest がない場合に、movie_preprocess 配下のファイルを直接走査して推論候補を作ります。
# - 最低限 sample_id/category/environment/path を復元するフォールバックです。

def scan_movie_preprocess_inventory() -> pd.DataFrame:
    # manifest がない場合の保険として、movie_preprocess 配下の *_front.mp4 を直接走査します。
    rows = []
    base_dir = PROJECT_ROOT / "data" / "movie_preprocess"
    for front_path in sorted(base_dir.glob("*/*/*_front.mp4")):
        sample_id = front_path.name[: -len("_front.mp4")]
        category = front_path.parent.parent.name
        environment = front_path.parent.name
        paths = build_preprocess_paths(sample_id, category, environment)
        rows.append({
            "sample_id": sample_id,
            "category": category,
            "environment": environment,
            **paths,
            "input_duration_sec": np.nan,
            "status": "scanned_from_files",
            "audio_status": "scanned_from_files",
        })
    return pd.DataFrame(rows)


# 関数メモ: load_movie_preprocess_inventory
# - manifest またはファイル走査から、推論候補の正規化された一覧表を作ります。
# - front/rear/audio が全て存在するかを usable として判定します。

def load_movie_preprocess_inventory(manifest_path: Path = MOVIE_PREPROCESS_MANIFEST_PATH) -> pd.DataFrame:
    # 推論候補の正本です。category / environment はここから自動解決します。
    if manifest_path.exists():
        manifest_df = pd.read_csv(manifest_path)
        rows = []
        for _, row in manifest_df.iterrows():
            category = str(row.get("category", "unknown"))
            environment = str(row.get("environment", "unknown"))
            sample_id = infer_sample_id_from_preprocess_path(row.get("front_output_path", row.get("input_video_path", "")))
            if not sample_id or sample_id.lower() == "nan":
                continue
            paths = build_preprocess_paths(sample_id, category, environment)
            rows.append({
                "sample_id": str(sample_id),
                "category": category,
                "environment": environment,
                **paths,
                "input_duration_sec": row.get("input_duration_sec", np.nan),
                "status": row.get("status", "unknown"),
                "audio_status": row.get("audio_status", "unknown"),
            })
        inventory_df = pd.DataFrame(rows)
    else:
        warnings.warn(f"movie_preprocess manifest not found. scanning files instead: {manifest_path}")
        inventory_df = scan_movie_preprocess_inventory()

    if inventory_df.empty:
        raise FileNotFoundError("No movie_preprocess samples were found.")

    for col in ["sample_id", "category", "environment"]:
        inventory_df[col] = inventory_df[col].astype(str).str.strip()
    for col in ["front_path", "rear_path", "audio_path"]:
        inventory_df[col] = inventory_df[col].map(Path)

    # front/rear/audio がそろっているものだけを usable とします。
    inventory_df["usable"] = inventory_df[["front_path", "rear_path", "audio_path"]].apply(
        lambda row: all(Path(value).exists() for value in row), axis=1
    )
    return inventory_df.drop_duplicates(["sample_id", "category", "environment"]).reset_index(drop=True)


# 関数メモ: load_train_sample_ids
# - 学習時に保存した sample_id 一覧を読み込みます。
# - random_untrained 推論で、学習に使った動画を評価対象から外すために使います。

def load_train_sample_ids(sample_list_path: Path = TRAIN_SAMPLE_LIST_PATH) -> set[str]:
    # ランダム抽出モードでは、学習に使った sample_id を除外します。
    if not sample_list_path.exists():
        warnings.warn(f"train sample list not found. random_untrained mode will not exclude trained samples: {sample_list_path}")
        return set()
    train_df = pd.read_csv(sample_list_path, dtype={"sample_id": str})
    if "sample_id" not in train_df.columns:
        warnings.warn(f"train sample list has no sample_id column: {sample_list_path}")
        return set()
    return set(train_df["sample_id"].dropna().astype(str).str.strip())


# 関数メモ: build_target_sample_from_inventory
# - inventory の1行を、推論処理で使う Series 形式へ変換します。
# - 出力ディレクトリや表示ラベルもここで作り、後続処理を単純にします。

def build_target_sample_from_inventory(row: pd.Series, label: str | None = None) -> pd.Series:
    sample_id = str(row["sample_id"])
    category = str(row["category"])
    environment = str(row["environment"])
    output_dir = TARGET_OUTPUT_ROOT / sample_id
    output_dir.mkdir(parents=True, exist_ok=True)
    short_id = sample_id.split("_", 1)[0]
    return pd.Series({
        "sample_id": sample_id,
        "category": category,
        "environment": environment,
        "plot_label": label or f"{category}/{environment}/{short_id}",
        "front_path": Path(row["front_path"]),
        "rear_path": Path(row["rear_path"]),
        "audio_path": Path(row["audio_path"]),
        "output_dir": output_dir,
    })


def build_target_sample(spec: dict, inventory_df: pd.DataFrame) -> pd.Series:
    # manual モードでは sample_id だけを指定し、category/environment/path は inventory から解決します。
    sample_id = str(spec["sample_id"]).strip()
    matches = inventory_df[inventory_df["sample_id"].astype(str).eq(sample_id)].copy()
    if "category" in spec:
        matches = matches[matches["category"].astype(str).eq(str(spec["category"]))]
    if "environment" in spec:
        matches = matches[matches["environment"].astype(str).eq(str(spec["environment"]))]
    if matches.empty:
        raise FileNotFoundError(f"sample_id was not found in movie_preprocess inventory: {sample_id}")
    usable_matches = matches[matches["usable"]]
    if usable_matches.empty:
        missing = matches.iloc[0][["front_path", "rear_path", "audio_path"]].tolist()
        raise FileNotFoundError(f"movie_preprocess files are incomplete for {sample_id}:\n" + "\n".join(map(str, missing)))
    if len(usable_matches) > 1:
        choices = usable_matches[["sample_id", "category", "environment"]].astype(str).agg("/".join, axis=1).tolist()
        raise ValueError("sample_id matched multiple movie_preprocess rows. Add category/environment to TARGET_SPECS:\n" + "\n".join(choices))
    return build_target_sample_from_inventory(usable_matches.iloc[0], label=spec.get("label"))


# 関数メモ: normalize_ratio_dict
# - 指定された比率辞書を、対象ラベルだけの合計1.0の比率へ正規化します。
# - 未指定ラベルや合計0の設定でも、ランダム抽出が破綻しないようにします。

def normalize_ratio_dict(ratio: dict[str, float], labels: list[str]) -> dict[str, float]:
    # 比率は合計1でなくてもよいので、正の値だけを取り出して正規化します。
    values = {label: max(float(ratio.get(label, 0.0)), 0.0) for label in labels}
    total = sum(values.values())
    if total <= 0:
        return {label: 1.0 / max(len(labels), 1) for label in labels}
    return {label: value / total for label, value in values.items()}


# 関数メモ: allocate_counts_by_ratio
# - 目標件数を比率に従って整数件数へ配分します。
# - 候補数が不足するラベルは上限で止め、余った枠を他ラベルへ再配分します。

def allocate_counts_by_ratio(available_counts: dict[str, int], ratio: dict[str, float], target_count: int) -> dict[str, int]:
    # 指定比率に近い整数件数へ変換します。候補数が足りないラベルは候補数で頭打ちにし、
    # 余った枠は同じ階層内のまだ余力があるラベルへ配ります。
    available_counts = {str(label): int(count) for label, count in available_counts.items() if int(count) > 0}
    if not available_counts:
        return {}
    target_count = min(int(target_count), sum(available_counts.values()))
    ratios = normalize_ratio_dict(ratio, sorted(available_counts))
    raw_targets = {label: target_count * ratios.get(label, 0.0) for label in available_counts}
    quotas = {label: min(int(math.floor(raw_targets[label])), available_counts[label]) for label in available_counts}

    remaining = target_count - sum(quotas.values())
    while remaining > 0:
        expandable = [label for label in available_counts if quotas[label] < available_counts[label]]
        if not expandable:
            break
        expandable.sort(
            key=lambda label: (raw_targets[label] - quotas[label], ratios.get(label, 0.0), available_counts[label]),
            reverse=True,
        )
        quotas[expandable[0]] += 1
        remaining -= 1
    return {label: count for label, count in quotas.items() if count > 0}


# 関数メモ: allocate_group_counts
# - random_untrained の目標件数を category と environment の2段階比率に分配します。
# - normal/anomaly と indoor/outdoor のバランスを同時に近づけるための配分処理です。

def allocate_group_counts(candidates_df: pd.DataFrame, target_count: int) -> dict[tuple[str, str], int]:
    # まず normal/anomaly の件数を RANDOM_TARGET_CATEGORY_RATIOS に合わせます。
    # その後、各 category の中で indoor/outdoor 比率に寄せます。
    # こうすると normal/outdoor が未学習候補に存在しない場合でも、normal 枠は normal/indoor へ再配分されます。
    target_count = min(int(target_count), len(candidates_df))
    category_available = candidates_df.groupby("category").size().to_dict()
    category_quotas = allocate_counts_by_ratio(category_available, RANDOM_TARGET_CATEGORY_RATIOS, target_count)

    quotas: dict[tuple[str, str], int] = {}
    for category, category_count in category_quotas.items():
        category_df = candidates_df[candidates_df["category"].astype(str).eq(str(category))]
        environment_available = category_df.groupby("environment").size().to_dict()
        environment_quotas = allocate_counts_by_ratio(environment_available, RANDOM_TARGET_ENVIRONMENT_RATIOS, category_count)
        for environment, count in environment_quotas.items():
            quotas[(str(category), str(environment))] = int(count)
    return {key: count for key, count in quotas.items() if count > 0}


# 関数メモ: select_random_untrained_targets
# - usable かつ学習に使っていない推論候補から、設定比率に従ってランダム抽出します。
# - 同じ random_state なら同じ対象が選ばれるよう、グループ別に安定した seed を使います。

def select_random_untrained_targets(inventory_df: pd.DataFrame) -> pd.DataFrame:
    train_sample_ids = load_train_sample_ids()
    candidates = inventory_df[inventory_df["usable"]].copy()
    candidates = candidates[~candidates["sample_id"].astype(str).isin(train_sample_ids)].reset_index(drop=True)
    if candidates.empty:
        raise ValueError("No usable movie_preprocess samples remain after excluding training samples.")

    if int(RANDOM_TARGET_COUNT) < 0:
        selected = candidates.sample(frac=1.0, random_state=RANDOM_TARGET_RANDOM_STATE).reset_index(drop=True)
        print(f"random_untrained: RANDOM_TARGET_COUNT < 0, using all untrained samples: {len(selected)}")
        return selected

    target_count = min(int(RANDOM_TARGET_COUNT), len(candidates))
    quotas = allocate_group_counts(candidates, target_count)
    selected_parts = []
    for index, ((category, environment), count) in enumerate(sorted(quotas.items())):
        group = candidates[candidates["category"].eq(category) & candidates["environment"].eq(environment)]
        # group ごとに seed を変えて、同じ設定なら常に同じサンプルが選ばれるようにします。
        stable_offset = sum((i + 1) * ord(ch) for i, ch in enumerate(f"{category}/{environment}"))
        selected_parts.append(group.sample(n=count, random_state=RANDOM_TARGET_RANDOM_STATE + stable_offset + index))

    selected = pd.concat(selected_parts, ignore_index=True) if selected_parts else candidates.head(0).copy()
    return selected.sample(frac=1.0, random_state=RANDOM_TARGET_RANDOM_STATE).reset_index(drop=True)


TARGET_INVENTORY_DF = load_movie_preprocess_inventory()
print("movie_preprocess usable counts:")
display(TARGET_INVENTORY_DF.groupby(["category", "environment", "usable"]).size().reset_index(name="count"))

if TARGET_SELECTION_MODE == "manual":
    if not TARGET_SPECS:
        raise ValueError("TARGET_SPECS must contain at least one target when TARGET_SELECTION_MODE='manual'.")
    TARGET_SAMPLES = [build_target_sample(spec, TARGET_INVENTORY_DF) for spec in TARGET_SPECS]
elif TARGET_SELECTION_MODE == "random_untrained":
    selected_targets_df = select_random_untrained_targets(TARGET_INVENTORY_DF)
    print("random_untrained selected counts:")
    display(selected_targets_df.groupby(["category", "environment"]).size().reset_index(name="selected_count"))
    TARGET_SAMPLES = [build_target_sample_from_inventory(row) for _, row in selected_targets_df.iterrows()]
else:
    raise ValueError("TARGET_SELECTION_MODE must be 'manual' or 'random_untrained'.")

for sample in TARGET_SAMPLES:
    missing = [str(sample[c]) for c in ["front_path", "rear_path", "audio_path"] if not Path(sample[c]).exists()]
    if missing:
        raise FileNotFoundError(f"Missing files for {sample['sample_id']}:\n" + "\n".join(missing))

print(f"target selection mode: {TARGET_SELECTION_MODE}")
print(f"targets: {len(TARGET_SAMPLES)}")
for sample in TARGET_SAMPLES[:TARGET_PRINT_LIMIT]:
    print(f"- {sample['plot_label']}: {sample['sample_id']}")
if len(TARGET_SAMPLES) > TARGET_PRINT_LIMIT:
    print(f"... {len(TARGET_SAMPLES) - TARGET_PRINT_LIMIT} more targets")


movie_preprocess usable counts:


,category,environment,usable,count
0,anomaly,indoor,True,87
1,anomaly,outdoor,True,5
2,normal,indoor,True,83
3,normal,outdoor,True,18


random_untrained selected counts:


,category,environment,selected_count
0,anomaly,indoor,20
1,anomaly,outdoor,5
2,normal,indoor,17
3,normal,outdoor,8


target selection mode: random_untrained
targets: 50
- anomaly/indoor/084: 084_旋回時、柱に衝突
- normal/indoor/1030: 1030
- normal/indoor/1033: 1033
- normal/outdoor/1017: 1017
- anomaly/indoor/043: 043_ラック下を通過時マストが衝突
- normal/outdoor/1015: 1015
- normal/indoor/1063: 1063
- normal/indoor/1062: 1062
- normal/indoor/1058: 1058
- anomaly/indoor/019: 019_ラックに衝突
- anomaly/indoor/039: 039_前進時、隣接荷物と接触し崩落ヒヤリ
- anomaly/indoor/088: 088_前進時、筋交いに追突
- normal/indoor/1089: 1089
- anomaly/indoor/054: 054_荷が崩落
- anomaly/indoor/086: 086_前進時、ラックに衝突
- anomaly/indoor/013: 013_荷卸し時仕切り版が落下
- normal/indoor/1099: 1099
- normal/outdoor/1005: 1005
- normal/outdoor/1018: 1018
- anomaly/indoor/008: 008_人身ヒヤリ
- anomaly/indoor/023: 023_旋回時、柱に衝突
- anomaly/indoor/073: 073_旋回時、ラックに衝突
- anomaly/outdoor/005: 005_後進旋回時トラックに衝突
- normal/indoor/1071: 1071
- normal/indoor/1041: 1041
- anomaly/indoor/065: 065_積荷がラックに衝突
- normal/outdoor/1010: 1010
- normal/indoor/1048: 1048
- normal/indoor/1040: 1040
- anomaly/indoor/030: 030_旋回時、安全ポール

## 3. 特徴量抽出

In [3]:
# ------------------------------------------------------------
# セル概要: 動画・音声読み込み
# ------------------------------------------------------------
# - 推論対象の動画と音声を、学習時と同じ窓幅・サンプリング条件で特徴化します。
# - 学習時と列定義がずれないよう、artifact の settings を優先しています。
# ------------------------------------------------------------

# ============================================================
# 動画・音声の読み込み
# ------------------------------------------------------------
# 動画は一定 FPS で間引き、必要なら横幅を縮小して処理量を抑えます。
# 音声は 0.2 秒窓へ分割し、RMS や log-mel などの特徴を作ります。
# ============================================================
# アスペクト比を保ったまま横幅だけをそろえます。flow の閾値を動画間で比較しやすくするためです。
# 関数メモ: resize_keep_aspect
# - アスペクト比を保ったまま横幅だけを縮小します。
# - 動画ごとの見た目を壊さず、optical flow の処理量とスケールをそろえます。

def resize_keep_aspect(frame: np.ndarray, width: int | None) -> np.ndarray:
    if width is None or frame.shape[1] <= width:
        return frame
    scale = width / frame.shape[1]
    height = max(1, int(round(frame.shape[0] * scale)))
    return cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)


# 関数メモ: extract_video_frames
# - 動画を指定FPS相当で間引いて読み込み、時刻付きフレームリストを作ります。
# - 音声や広域振動スコアと同じ時間軸へ後で結合するため、各フレームに `time` を持たせます。

def extract_video_frames(video_path: str | Path, fps_sample: float = FPS_SAMPLE, resize_width: int | None = FRAME_RESIZE_WIDTH) -> list[dict]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        warnings.warn(f"Could not open video: {video_path}")
        return []
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    if not src_fps or np.isnan(src_fps) or src_fps <= 0:
        src_fps = fps_sample
    interval = max(1, int(round(src_fps / fps_sample)))

    frames = []
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_idx % interval == 0:
            frames.append({"time": frame_idx / src_fps, "frame": resize_keep_aspect(frame, resize_width)})
        frame_idx += 1
    cap.release()
    return frames


# 関数メモ: load_audio_mono
# - wav をモノラル float32 に変換し、必要なら指定サンプリング周波数へリサンプリングします。
# - 学習時と推論時で音声特徴の窓幅・周波数軸を一致させるための入口です。

def load_audio_mono(wav_path: str | Path, sr: int = AUDIO_SR) -> tuple[np.ndarray, int]:
    if librosa is not None:
        y, used_sr = librosa.load(wav_path, sr=sr, mono=True)
        return y.astype(np.float32), used_sr
    used_sr, y = wavfile.read(wav_path)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    max_abs = np.max(np.abs(y)) if y.size else 1.0
    if max_abs > 0:
        y = y / max_abs
    if used_sr != sr and y.size:
        gcd = math.gcd(used_sr, sr)
        y = resample_poly(y, sr // gcd, used_sr // gcd).astype(np.float32)
        used_sr = sr
    return y, used_sr


# 関数メモ: extract_audio_features
# - 音声を固定時間窓に分け、RMS・ピーク・ZCR・周波数統計・log-mel を作ります。
# - 音声由来の異常度を学習・推論するための基本特徴表を返します。

def extract_audio_features(wav_path: str | Path, video_id: str) -> pd.DataFrame:
    y, used_sr = load_audio_mono(wav_path, sr=AUDIO_SR)
    win = max(1, int(round(WINDOW_SEC * used_sr)))
    if y.size == 0:
        return pd.DataFrame(columns=["video_id", "time_bin", "time", "audio_missing"])

    rows = []
    for i in range(int(math.ceil(len(y) / win))):
        segment = y[i * win : min(len(y), (i + 1) * win)]
        if segment.size == 0:
            continue
        spectrum = np.abs(np.fft.rfft(segment * np.hanning(segment.size)))
        freqs = np.fft.rfftfreq(segment.size, d=1.0 / used_sr)
        power = spectrum ** 2
        power_sum = float(power.sum()) + 1e-12
        centroid = float((freqs * power).sum() / power_sum)
        rows.append({
            "video_id": video_id,
            "time_bin": i,
            "time": i * WINDOW_SEC,
            "audio_rms": float(np.sqrt(np.mean(segment ** 2))),
            "audio_energy": float(np.mean(segment ** 2)),
            "audio_peak": float(np.max(np.abs(segment))),
            "audio_ptp": float(np.ptp(segment)),
            "audio_zcr": float(np.mean(np.abs(np.diff(np.signbit(segment))).astype(float))) if segment.size > 1 else 0.0,
            "audio_centroid": centroid,
            "audio_bandwidth": float(np.sqrt((((freqs - centroid) ** 2) * power).sum() / power_sum)),
            "audio_high_freq_energy": float(power[freqs >= 3000].sum() / power_sum),
            "audio_missing": 0,
        })
    df = pd.DataFrame(rows)

    if librosa is not None and len(df):
        mel = librosa.feature.melspectrogram(
            y=y, sr=used_sr, n_mels=N_MELS, n_fft=min(2048, max(256, win)), hop_length=win, power=2.0
        )
        log_mel = librosa.power_to_db(mel, ref=np.max).T
        for m in range(N_MELS):
            values = np.full(len(df), np.nan, dtype=float)
            values[: min(len(values), log_mel.shape[0])] = log_mel[: min(len(values), log_mel.shape[0]), m]
            df[f"audio_mel_{m:02d}"] = values
    else:
        for m in range(N_MELS):
            df[f"audio_mel_{m:02d}"] = 0.0
    return df

In [4]:
# ------------------------------------------------------------
# セル概要: 推論用 optical flow 特徴
# ------------------------------------------------------------
# - Farneback optical flow を計算し、全体統計とグリッド別 x/y/magnitude を作ります。
# - ガウシアンブラーや有効領域マスクはここで制御し、推論側のノイズ対策を調整できます。
# ------------------------------------------------------------

# ============================================================
# optical flow 特徴量
# ------------------------------------------------------------
# Farneback optical flow で、各フレーム間の画素移動量を推定します。
# 後段の広域振動スコアでは、3x3 グリッドごとの flow x/y 平均を使い、
# 向きが小刻みに変化しているかを評価します。
# ============================================================
# 今はマスクなしで画面全体を有効領域として扱います。必要になればここで領域除外できます。
# 関数メモ: build_motion_valid_mask
# - flow 集計に使う有効領域マスクを作ります。
# - 現在は全領域を有効にしていますが、車体や固定UIを除外したい場合の差し替え点です。

def build_motion_valid_mask(frame_shape: tuple[int, ...], prefix: str | None = None) -> np.ndarray:
    h, w = int(frame_shape[0]), int(frame_shape[1])
    return np.ones((h, w), dtype=bool)


# 関数メモ: masked_values
# - 有効マスク内の値だけを取り出し、有限値に絞ります。
# - マスクが空の場合は必要に応じて全画素へフォールバックし、統計値が欠落しにくくします。

def masked_values(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> np.ndarray:
    selected = values[valid_mask]
    if selected.size == 0 and fallback_to_all:
        selected = values.reshape(-1)
    return selected[np.isfinite(selected)]


# 関数メモ: masked_mean
# - マスク済み値の平均を返します。
# - 対象領域が空でも 0.0 を返し、特徴量テーブルの列構造を保ちます。

def masked_mean(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> float:
    selected = masked_values(values, valid_mask, fallback_to_all=fallback_to_all)
    return float(np.mean(selected)) if selected.size else 0.0


# 関数メモ: masked_std
# - マスク済み値の標準偏差を返します。
# - flow のばらつきを特徴化しつつ、空領域では安全に 0.0 にします。

def masked_std(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> float:
    selected = masked_values(values, valid_mask, fallback_to_all=fallback_to_all)
    return float(np.std(selected)) if selected.size else 0.0


# 関数メモ: masked_max
# - マスク済み値の最大値を返します。
# - 一部だけ強く動いた場合のピーク量を拾うために使います。

def masked_max(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> float:
    selected = masked_values(values, valid_mask, fallback_to_all=fallback_to_all)
    return float(np.max(selected)) if selected.size else 0.0


# 関数メモ: normalize_gaussian_blur_kernel
# - 設定値から OpenCV が受け付ける奇数サイズのガウシアンカーネルへ正規化します。
# - 0や1以下ならブラーなし扱いにして、設定ミスでエラーになりにくくします。

def normalize_gaussian_blur_kernel(kernel: tuple[int, int] | list[int] | int | float) -> tuple[int, int]:
    if isinstance(kernel, (int, float)):
        kx = ky = int(kernel)
    else:
        values = list(kernel)
        if not values:
            return 0, 0
        kx = int(values[0])
        ky = int(values[1]) if len(values) > 1 else kx
    if kx <= 1 or ky <= 1:
        return 0, 0
    return kx + (kx % 2 == 0), ky + (ky % 2 == 0)


# 関数メモ: apply_flow_gaussian_blur
# - optical flow 前のグレースケール画像へ必要に応じてガウシアンブラーをかけます。
# - 細かいノイズによる flow のちらつきを抑えるための推論側オプションです。

def apply_flow_gaussian_blur(gray: np.ndarray) -> np.ndarray:
    kx, ky = normalize_gaussian_blur_kernel(FLOW_GAUSSIAN_BLUR_KERNEL)
    if kx <= 1 or ky <= 1:
        return gray
    return cv2.GaussianBlur(gray, (kx, ky), sigmaX=FLOW_GAUSSIAN_BLUR_SIGMA, sigmaY=FLOW_GAUSSIAN_BLUR_SIGMA)


# Farneback の計算負荷を下げるため、解析用グレースケール画像を縮小します。
# 関数メモ: resize_gray_for_flow
# - flow 計算用のグレースケール画像を縮小し、元サイズ換算に戻すための倍率も返します。
# - Farneback の計算負荷を抑えつつ、出力 flow を元のリサイズ済み動画スケールで解釈します。

def resize_gray_for_flow(gray: np.ndarray, scale: float = FLOW_ANALYSIS_SCALE) -> tuple[np.ndarray, float, float]:
    scale = float(scale)
    if scale <= 0.0 or scale >= 1.0:
        return apply_flow_gaussian_blur(gray), 1.0, 1.0
    h, w = gray.shape[:2]
    new_w = max(8, int(round(w * scale)))
    new_h = max(8, int(round(h * scale)))
    if new_w == w and new_h == h:
        return apply_flow_gaussian_blur(gray), 1.0, 1.0
    resized = cv2.resize(gray, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return apply_flow_gaussian_blur(resized), new_w / max(w, 1), new_h / max(h, 1)


# flow 計算に失敗したフレームでも列構造を崩さないため、0 埋め行を返します。
# 関数メモ: _empty_flow_row
# - flow 計算に失敗した時刻でも、期待される列を0埋めで返します。
# - 一部フレームの失敗で DataFrame の列が欠け、後続の結合やモデル入力が壊れるのを防ぎます。

def _empty_flow_row(prefix: str, time_sec: float, time_bin: int, video_id: str | None) -> dict:
    row = {
        "video_id": video_id,
        "time_bin": time_bin,
        "time": time_sec,
        f"{prefix}_flow_mag_mean": 0.0,
        f"{prefix}_flow_mag_std": 0.0,
        f"{prefix}_flow_mag_max": 0.0,
        f"{prefix}_flow_angle_mean": 0.0,
        f"{prefix}_flow_angle_std": 0.0,
        f"{prefix}_flow_x_mean": 0.0,
        f"{prefix}_flow_x_std": 0.0,
        f"{prefix}_flow_y_mean": 0.0,
        f"{prefix}_flow_y_std": 0.0,
        f"{prefix}_flow_failed": 1.0,
    }
    for gy in range(FLOW_GRID[0]):
        for gx in range(FLOW_GRID[1]):
            for name in ["mag_mean", "mag_std", "x_mean", "y_mean", "valid_ratio"]:
                row[f"{prefix}_flow_cell_{gy}_{gx}_{name}"] = 0.0
    return row


# 前フレームと現フレームの差分から、全体平均と 3x3 グリッド別の flow を作ります。
# 関数メモ: compute_optical_flow_features
# - 連続フレーム間の dense optical flow を計算し、全体統計とグリッド統計へ集約します。
# - front/rear それぞれの motion 特徴と、後段の広域振動スコアの材料を作ります。

def compute_optical_flow_features(
    frames: list[dict],
    prefix: str,
    window_sec: float = WINDOW_SEC,
    grid: tuple[int, int] = FLOW_GRID,
    video_id: str | None = None,
) -> pd.DataFrame:
    if len(frames) < 2:
        return pd.DataFrame()

    rows = []
    prev_gray, prev_scale_x, prev_scale_y = resize_gray_for_flow(cv2.cvtColor(frames[0]["frame"], cv2.COLOR_BGR2GRAY))
    for item in frames[1:]:
        time_sec = float(item["time"])
        time_bin = int(round(time_sec / window_sec))
        try:
            gray, scale_x, scale_y = resize_gray_for_flow(cv2.cvtColor(item["frame"], cv2.COLOR_BGR2GRAY))
                        # Farneback は各画素の移動ベクトルを dense に推定します。
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, gray, None,
                pyr_scale=0.5, levels=3, winsize=15, iterations=3,
                poly_n=5, poly_sigma=1.2, flags=0,
            )
                        # 縮小画像で計算した移動量を、元のリサイズ済み画像スケールへ戻します。
            fx = flow[..., 0] / max(scale_x, 1e-6)
            fy = flow[..., 1] / max(scale_y, 1e-6)
            mag, angle = cv2.cartToPolar(fx, fy, angleInDegrees=False)
            valid_mask = build_motion_valid_mask(mag.shape, prefix)
            row = {
                "video_id": video_id,
                "time_bin": time_bin,
                "time": time_sec,
                f"{prefix}_flow_mag_mean": masked_mean(mag, valid_mask),
                f"{prefix}_flow_mag_std": masked_std(mag, valid_mask),
                f"{prefix}_flow_mag_max": masked_max(mag, valid_mask),
                f"{prefix}_flow_angle_mean": masked_mean(angle, valid_mask),
                f"{prefix}_flow_angle_std": masked_std(angle, valid_mask),
                f"{prefix}_flow_x_mean": masked_mean(fx, valid_mask),
                f"{prefix}_flow_x_std": masked_std(fx, valid_mask),
                f"{prefix}_flow_y_mean": masked_mean(fy, valid_mask),
                f"{prefix}_flow_y_std": masked_std(fy, valid_mask),
                f"{prefix}_flow_failed": 0.0,
            }
            h, w = mag.shape
                        # 画面を 3x3 に分け、どの領域で振動が強いかを後から見られるようにします。
            for gy in range(grid[0]):
                y0, y1 = int(h * gy / grid[0]), int(h * (gy + 1) / grid[0])
                for gx in range(grid[1]):
                    x0, x1 = int(w * gx / grid[1]), int(w * (gx + 1) / grid[1])
                    cell_mask = valid_mask[y0:y1, x0:x1]
                    row[f"{prefix}_flow_cell_{gy}_{gx}_mag_mean"] = masked_mean(mag[y0:y1, x0:x1], cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_mag_std"] = masked_std(mag[y0:y1, x0:x1], cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_x_mean"] = masked_mean(fx[y0:y1, x0:x1], cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_y_mean"] = masked_mean(fy[y0:y1, x0:x1], cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_valid_ratio"] = float(np.mean(cell_mask)) if cell_mask.size else 0.0
        except Exception as exc:
            warnings.warn(f"{prefix} optical flow failed at {time_sec:.2f}s: {exc}")
            gray, scale_x, scale_y = resize_gray_for_flow(cv2.cvtColor(item["frame"], cv2.COLOR_BGR2GRAY))
            row = _empty_flow_row(prefix, time_sec, time_bin, video_id)
        rows.append(row)
        prev_gray, prev_scale_x, prev_scale_y = gray, scale_x, scale_y

    return pd.DataFrame(rows)


In [5]:
# ------------------------------------------------------------
# セル概要: 時間結合と広域振動特徴
# ------------------------------------------------------------
# - 音声特徴、raw flow、広域振動スコアを time_bin で結合します。
# - motion モデルが使う broad vibration 列を、学習時と同じ手順で再生成します。
# ------------------------------------------------------------

# ============================================================
# 広域振動スコアと特徴量結合
# ------------------------------------------------------------
# audio / flow など別々に作った特徴を時間窓で結合し、
# raw flow から「同時刻の上位グリッド平均」である広域振動スコアを作ります。
# motion モデルにはこの front/rear の広域振動スコアだけを入れます。
# ============================================================
# 各特徴ソースを video_id + time_bin ごとに1行へ集約します。
# 関数メモ: aggregate_feature_df
# - 任意の特徴表を video_id + time_bin 単位へ集約します。
# - 推論ノート側で音声・flow・広域振動を同じ時間粒度にそろえるために使います。

def aggregate_feature_df(df: pd.DataFrame, window_sec: float = WINDOW_SEC) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    df = df.copy()
    if "video_id" not in df.columns:
        df["video_id"] = "unknown"
    if "time_bin" not in df.columns:
        if "time" not in df.columns:
            raise ValueError("feature df must have either time_bin or time")
        df["time_bin"] = np.round(df["time"] / window_sec).astype(int)

    key_cols = ["video_id", "time_bin"]
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in {"time", "time_bin"}]
    other_cols = [c for c in df.columns if c not in set(key_cols + numeric_cols + ["time"])]
    out = df.groupby(key_cols, as_index=False)[numeric_cols].mean() if numeric_cols else df[key_cols].drop_duplicates()
    for col in other_cols:
        values = df.groupby(key_cols)[col].agg(lambda x: x.dropna().mode().iat[0] if len(x.dropna().mode()) else "unknown").reset_index()
        out = out.merge(values, on=key_cols, how="left")
    out["time"] = out["time_bin"] * window_sec
    return out


# 音声・flow・広域振動スコアを time_bin で outer join し、欠損は前後の値で補います。
# 関数メモ: align_features_by_time
# - 複数の特徴表を time_bin で outer join し、欠損値を動画内で前後補完します。
# - 音声0.2秒窓と広域振動0.5秒hopのような異なる時間解像度を、モデル入力の1表へそろえます。

def align_features_by_time(feature_dfs: list[pd.DataFrame], window_sec: float = WINDOW_SEC) -> pd.DataFrame:
    valid = [aggregate_feature_df(df, window_sec=window_sec) for df in feature_dfs if df is not None and len(df)]
    if not valid:
        return pd.DataFrame()
    merged = valid[0]
    for df in valid[1:]:
        merged = merged.merge(df.drop(columns=["time"], errors="ignore"), on=["video_id", "time_bin"], how="outer")
    merged["time"] = merged["time_bin"] * window_sec
    merged = merged.sort_values(["video_id", "time_bin"]).reset_index(drop=True)
    numeric_cols = [c for c in merged.select_dtypes(include=[np.number]).columns if c not in {"time", "time_bin"}]
    merged[numeric_cols] = merged.groupby("video_id", group_keys=False)[numeric_cols].apply(lambda g: g.ffill().bfill()).fillna(0.0)
    for col in merged.select_dtypes(exclude=[np.number]).columns:
        if col != "video_id":
            merged[col] = merged[col].fillna("unknown")
    return merged


# 中央値と MAD を使い、外れ値に強い正方向 z-score を作ります。
# 関数メモ: robust_positive_zscore
# - 中央値とMADに基づく、正方向だけの頑健 z-score を作ります。
# - 通常より大きい変化だけを振動候補として強調し、小さい側の変動は0に丸めます。

def robust_positive_zscore(values: np.ndarray | pd.Series) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return np.zeros((0,), dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return np.zeros_like(values, dtype=float)
    median = float(np.median(finite))
    scale = max(1.4826 * float(np.median(np.abs(finite - median))), 1e-6)
    return np.maximum((np.nan_to_num(values, nan=median) - median) / scale, 0.0)


# スコアを指定パーセンタイル範囲で 0〜1 に正規化します。rear だけ極端に跳ねる問題を抑える目的です。
# 関数メモ: percentile_normalize_0_1
# - 指定パーセンタイル範囲を使ってスコアを0〜1へ正規化します。
# - 動画やカメラごとのスケール差を抑え、rear/front の値を比較しやすくします。

def percentile_normalize_0_1(
    values: np.ndarray | pd.Series,
    lower_percentile: float = DIRECTION_JITTER_SCORE_LOWER_PERCENTILE,
    upper_percentile: float = DIRECTION_JITTER_SCORE_UPPER_PERCENTILE,
    positive_only: bool = True,
) -> np.ndarray:
    values_arr = np.asarray(values, dtype=float)
    out = np.zeros_like(values_arr, dtype=float)
    finite_mask = np.isfinite(values_arr)
    if not finite_mask.any():
        return out
    fit_values = values_arr[finite_mask]
    if positive_only:
        fit_values = fit_values[fit_values > FLOW_SCORE_MIN_VISIBLE]
    if fit_values.size == 0:
        return out
    lo_q, hi_q = sorted([float(np.clip(lower_percentile, 0.0, 100.0)), float(np.clip(upper_percentile, 0.0, 100.0))])
    lo, hi = float(np.percentile(fit_values, lo_q)), float(np.percentile(fit_values, hi_q))
    if hi <= lo:
        out[finite_mask] = np.where(values_arr[finite_mask] > FLOW_SCORE_MIN_VISIBLE, 1.0, 0.0)
    else:
        out[finite_mask] = np.clip((values_arr[finite_mask] - lo) / max(hi - lo, 1e-6), 0.0, 1.0)
    return np.where(values_arr > FLOW_SCORE_MIN_VISIBLE, out, 0.0).astype(float) if positive_only else out.astype(float)


# 関数メモ: build_flow_window_starts
# - raw flow の長さに対して、広域振動スコア用の1秒窓開始時刻を作ります。
# - 短い動画でも最低1窓は返し、後続テーブルが空になりにくいようにします。

def build_flow_window_starts(duration_sec: float, window_sec: float = FLOW_SCORE_WINDOW_SEC, hop_sec: float = FLOW_SCORE_HOP_SEC) -> list[float]:
    max_start = max(float(duration_sec) - float(window_sec), 0.0)
    starts = np.arange(0.0, max_start + 1e-9, float(hop_sec), dtype=np.float32).tolist()
    return [float(value) for value in (starts if starts else [0.0])]


# flow ベクトルの向き変化量 |dtheta| を計算します。小さすぎるベクトルは向きが不安定なので0扱いします。
# 関数メモ: compute_flow_direction_change_abs
# - グリッドごとの flow x/y から、連続時刻の向き変化量 |dtheta| を計算します。
# - ベクトルが小さすぎる時刻は向きが不安定なので、振動スコアから除外します。

def compute_flow_direction_change_abs(raw_flow_df: pd.DataFrame, x_col: str, y_col: str) -> pd.Series:
    if x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.Series(dtype=float)
    ordered = raw_flow_df.sort_values("time")
    x = ordered[x_col].astype(float).to_numpy()
    y = ordered[y_col].astype(float).to_numpy()
    magnitude = np.hypot(x, y)
    angle = np.arctan2(y, x)
    delta = np.diff(angle, prepend=angle[:1])
    direction_change = np.abs(np.arctan2(np.sin(delta), np.cos(delta)))
    valid = (magnitude >= FLOW_DIRECTION_MIN_MAG) & (np.r_[magnitude[:1], magnitude[:-1]] >= FLOW_DIRECTION_MIN_MAG)
    return pd.Series(np.where(valid, np.clip(direction_change, 0.0, np.pi), 0.0), index=ordered.index).reindex(raw_flow_df.index).fillna(0.0)


# 振動候補の high/low 閾値を、固定値またはパーセンタイルから決定します。
# 関数メモ: resolve_direction_jitter_thresholds
# - 方向変化スコアから high/low 閾値を固定値またはパーセンタイルで決めます。
# - 動画ごとの分布に合わせて、振動イベント候補の感度を調整します。

def resolve_direction_jitter_thresholds(scores: np.ndarray | pd.Series) -> tuple[float, float]:
    scores_arr = np.asarray(scores, dtype=float)
    positive = scores_arr[np.isfinite(scores_arr) & (scores_arr > FLOW_SCORE_MIN_VISIBLE)]
    if DIRECTION_JITTER_THRESHOLD_MODE == "percentile" and positive.size:
        high = float(np.percentile(positive, np.clip(DIRECTION_JITTER_HIGH_PERCENTILE, 0.0, 100.0)))
        low = float(np.percentile(positive, np.clip(DIRECTION_JITTER_LOW_PERCENTILE, 0.0, 100.0)))
    else:
        high = float(DIRECTION_JITTER_HIGH_THRESHOLD)
        low = float(DIRECTION_JITTER_LOW_THRESHOLD)
    return high, min(low, high)


# high 閾値を超えた窓を起点に、隣接する low 閾値以上の窓も同じ振動イベントとして拾います。
# 関数メモ: mark_direction_jitter_hysteresis_windows
# - high 閾値を超えた窓を起点に、隣接する low 閾値以上の窓もイベントとして選びます。
# - ピーク1点だけでなく、その前後のまとまった振動区間を拾うためのヒステリシス処理です。

def mark_direction_jitter_hysteresis_windows(window_df: pd.DataFrame) -> pd.DataFrame:
    window_df = window_df.sort_values("window_start_sec").copy()
    scores = np.nan_to_num(window_df["direction_jitter_score"].to_numpy(dtype=float), nan=-np.inf)
    high, low = resolve_direction_jitter_thresholds(scores)
    seed_mask = (scores >= high) & (scores > FLOW_SCORE_MIN_VISIBLE)
    low_mask = (scores >= low) & (scores > FLOW_SCORE_MIN_VISIBLE)
    selected = np.zeros(len(window_df), dtype=bool)
    for seed_index in np.flatnonzero(seed_mask):
        selected[seed_index] = True
        left = seed_index - 1
        while left >= 0 and low_mask[left]:
            selected[left] = True
            left -= 1
        right = seed_index + 1
        while right < len(window_df) and low_mask[right]:
            selected[right] = True
            right += 1
    window_df["direction_jitter_seed"] = seed_mask.astype(bool)
    window_df["direction_jitter_selected"] = selected.astype(bool)
    window_df["direction_jitter_high_threshold"] = high
    window_df["direction_jitter_low_threshold"] = low
    return window_df


# 関数メモ: add_direction_jitter_score_alpha
# - 選択された振動窓に、可視化用の透明度 alpha を付与します。
# - スコアが高いほど濃く描けるようにし、候補区間の強弱を図へ反映します。

def add_direction_jitter_score_alpha(window_df: pd.DataFrame) -> pd.DataFrame:
    if window_df.empty or "direction_jitter_score" not in window_df.columns:
        window_df = window_df.copy()
        window_df["direction_jitter_seed"] = False
        window_df["direction_jitter_selected"] = False
        window_df["direction_jitter_score_alpha"] = 0.0
        return window_df
    window_df = mark_direction_jitter_hysteresis_windows(window_df)
    selected_scores = window_df.loc[window_df["direction_jitter_selected"], "direction_jitter_score"].to_numpy(dtype=float)
    finite = selected_scores[np.isfinite(selected_scores)]
    if finite.size == 0 or float(np.nanmax(finite)) <= FLOW_SCORE_MIN_VISIBLE:
        window_df["direction_jitter_score_alpha"] = 0.0
        return window_df
    lo, hi = float(np.nanmin(finite)), float(np.nanmax(finite))
    scores = window_df["direction_jitter_score"].to_numpy(dtype=float)
    normalized = np.where(window_df["direction_jitter_selected"].to_numpy(dtype=bool), 1.0, 0.0) if hi <= lo else np.clip((np.nan_to_num(scores, nan=lo) - lo) / max(hi - lo, 1e-6), 0.0, 1.0)
    window_df["direction_jitter_score_alpha"] = np.where(
        window_df["direction_jitter_selected"].to_numpy(dtype=bool),
        FLOW_SCORE_ALPHA_MIN + normalized * (FLOW_SCORE_ALPHA_MAX - FLOW_SCORE_ALPHA_MIN),
        0.0,
    ).astype(float)
    return window_df


# 1グリッドについて、1秒窓ごとの向き変化統計と振動スコアを作ります。
# 関数メモ: build_flow_direction_jitter_window_table
# - 1つの camera/grid について、1秒窓ごとの方向変化統計と振動スコアを作ります。
# - 複数指標を合成して direction_jitter_score を作り、グリッド別の局所振動を評価します。

def build_flow_direction_jitter_window_table(raw_flow_df: pd.DataFrame, camera: str, gy: int, gx: int) -> pd.DataFrame:
    x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
    y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
    if raw_flow_df.empty or x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.DataFrame()

    ordered = raw_flow_df.sort_values("time").reset_index(drop=True)
    direction_change = compute_flow_direction_change_abs(ordered, x_col, y_col).to_numpy(dtype=float)
    times = ordered["time"].astype(float).to_numpy()
    if times.size == 0:
        return pd.DataFrame()
    duration_sec = max(float(np.nanmax(times)) if np.isfinite(times).any() else 0.0, FLOW_SCORE_WINDOW_SEC)

    rows = []
    for start_sec in build_flow_window_starts(duration_sec):
        end_sec = float(min(start_sec + FLOW_SCORE_WINDOW_SEC, duration_sec))
        start_sec = max(end_sec - FLOW_SCORE_WINDOW_SEC, 0.0)
        mask = (times >= start_sec) & (times < end_sec)
        if not np.any(mask):
            mask = np.zeros_like(times, dtype=bool)
            mask[int(np.argmin(np.abs(times - start_sec)))] = True
        segment = direction_change[mask]
        direction_max = float(np.max(segment))
        high_ratio = float(np.mean(segment >= direction_max * FLOW_SCORE_HIGH_RATIO_FRACTION)) if direction_max > FLOW_SCORE_MIN_VISIBLE else 0.0
        rows.append({
            "camera": camera,
            "grid_x": gx + 1,
            "grid_y": gy + 1,
            "grid_col": gx,
            "grid_row": gy,
            "window_start_sec": float(start_sec),
            "window_end_sec": float(end_sec),
            "window_center_sec": float(0.5 * (start_sec + end_sec)),
            "direction_mean": float(np.mean(segment)),
            "direction_sum": float(np.sum(segment)),
            "direction_p95": float(np.percentile(segment, 95)),
            "direction_max": direction_max,
            "direction_std": float(np.std(segment)),
            "direction_variation": float(np.mean(np.abs(np.diff(segment)))) if segment.size >= 2 else 0.0,
            "direction_high_ratio": high_ratio,
        })

    window_df = pd.DataFrame(rows)
    for name in ["direction_sum", "direction_high_ratio", "direction_variation", "direction_p95"]:
        window_df[f"{name}_z"] = robust_positive_zscore(window_df[name])
        # 複数の向き変化指標を重み付きで足し、ギザギザした揺れを1つの raw スコアにします。
    window_df["direction_jitter_score_raw"] = (
        0.35 * window_df["direction_sum_z"]
        + 0.25 * window_df["direction_high_ratio_z"]
        + 0.25 * window_df["direction_variation_z"]
        + 0.15 * window_df["direction_p95_z"]
    ).astype(float)
        # raw スコアは動画・カメラごとのスケール差が出やすいため、0〜1へ正規化します。
    window_df["direction_jitter_score"] = percentile_normalize_0_1(window_df["direction_jitter_score_raw"])
    return add_direction_jitter_score_alpha(window_df)


# 関数メモ: build_flow_direction_jitter_score_table
# - 指定 camera の全グリッドに対して direction jitter の窓表を作って結合します。
# - 3x3 のどこで振動が出たかを、camera 単位で一覧化します。

def build_flow_direction_jitter_score_table(raw_flow_df: pd.DataFrame, camera: str) -> pd.DataFrame:
    tables = [
        build_flow_direction_jitter_window_table(raw_flow_df, camera, gy, gx)
        for gy in range(FLOW_GRID[0])
        for gx in range(FLOW_GRID[1])
    ]
    tables = [df for df in tables if len(df)]
    return pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()


# 同じ時刻のグリッドをスコア順に並べ、上位 K 個を可視化・集計できるよう印を付けます。
# 関数メモ: add_direction_jitter_topk_columns
# - 同じ時刻のグリッドを振動スコア順に並べ、上位K件に rank と選択フラグを付けます。
# - 広域振動スコアを作るときに、画面全体のうち強く揺れた領域だけを平均するために使います。

def add_direction_jitter_topk_columns(direction_jitter_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    if direction_jitter_df is None or direction_jitter_df.empty:
        return pd.DataFrame()
    required = {"video_id", "camera", "window_start_sec", "direction_jitter_score"}
    missing = sorted(required - set(direction_jitter_df.columns))
    if missing:
        raise ValueError(f"Missing columns for direction jitter top-k: {missing}")

    work = direction_jitter_df.copy()
    selected = work.get("direction_jitter_selected", pd.Series(False, index=work.index)).astype(bool)
    scores = pd.to_numeric(work["direction_jitter_score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    work["thresholded_direction_jitter_score"] = np.where(selected, scores, 0.0)
    work["direction_jitter_topk_rank"] = pd.Series(pd.NA, index=work.index, dtype="Int64")
    work["direction_jitter_topk_selected"] = False
    work["window_start_bin"] = np.round(work["window_start_sec"].astype(float) / max(FLOW_SCORE_HOP_SEC, 1e-6)).astype(int)

    meta_cols = [c for c in ["target_label", "target_category", "target_environment"] if c in work.columns]
    for _, group in work.groupby(["video_id", *meta_cols, "camera", "window_start_bin"], sort=False):
        ranked = group[group["thresholded_direction_jitter_score"] > FLOW_SCORE_MIN_VISIBLE]
        ranked = ranked.sort_values("thresholded_direction_jitter_score", ascending=False).head(max(1, int(top_k)))
        if len(ranked):
            work.loc[ranked.index, "direction_jitter_topk_rank"] = np.arange(1, len(ranked) + 1, dtype=int)
            work.loc[ranked.index, "direction_jitter_topk_selected"] = True
    return work


# カメラごと・時刻ごとに、閾値通過後の上位 K グリッドを平均して広域振動スコアにします。
# 関数メモ: build_broad_vibration_score_table
# - camera/time ごとに上位Kグリッドの振動スコアを平均し、広域振動スコアを作ります。
# - 局所的な強い揺れを拾いつつ、単一グリッドのノイズに引っ張られすぎない形にします。

def build_broad_vibration_score_table(direction_jitter_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    if direction_jitter_df is None or direction_jitter_df.empty:
        return pd.DataFrame()
    work = add_direction_jitter_topk_columns(direction_jitter_df, top_k=top_k)
    meta_cols = [c for c in ["target_label", "target_category", "target_environment"] if c in work.columns]
    rows = []
    for keys, group in work.groupby(["video_id", *meta_cols, "camera", "window_start_bin"], sort=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        video_id = keys[0]
        meta_values = dict(zip(meta_cols, keys[1:1 + len(meta_cols)]))
        camera = keys[1 + len(meta_cols)]
        window_start_bin = keys[2 + len(meta_cols)]
        values = np.sort(np.nan_to_num(group["thresholded_direction_jitter_score"].to_numpy(dtype=float), nan=0.0))[::-1]
        top_values = np.zeros(max(1, int(top_k)), dtype=float)
        top_values[: min(top_values.size, values.size)] = values[: min(top_values.size, values.size)]
        rows.append({
            "video_id": video_id,
            **meta_values,
            "camera": camera,
            "window_start_bin": int(window_start_bin),
            "window_start_sec": float(group["window_start_sec"].min()),
            "window_end_sec": float(group["window_end_sec"].max()),
            "window_center_sec": float(group["window_center_sec"].median()),
            "broad_vibration_score": float(top_values.mean()),
            "broad_vibration_top_k": int(top_values.size),
            "selected_grid_count": int(group.get("direction_jitter_selected", pd.Series(False, index=group.index)).astype(bool).sum()),
            "nonzero_top_k_count": int(np.count_nonzero(top_values > FLOW_SCORE_MIN_VISIBLE)),
            "max_thresholded_direction_jitter_score": float(np.max(values)) if values.size else 0.0,
            **{f"broad_vibration_top{i}_score": float(v) for i, v in enumerate(top_values, start=1)},
        })
    return pd.DataFrame(rows).sort_values(["video_id", "camera", "window_start_sec"]).reset_index(drop=True)


# 広域振動スコア表を、モデル入力に使いやすい front/rear の2列へ pivot します。
# 関数メモ: build_broad_vibration_feature_df
# - grid別 direction jitter から、モデル入力用の front/rear 広域振動特徴列を作ります。
# - motion モデルへ入る主特徴である `front_broad_vibration_score` / `rear_broad_vibration_score` を生成します。

def build_broad_vibration_feature_df(raw_flow_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    empty_cols = ["video_id", "time", "time_bin", *BROAD_VIBRATION_COLUMNS]
    if raw_flow_df is None or raw_flow_df.empty:
        return pd.DataFrame(columns=empty_cols)

    raw_flow_df = raw_flow_df.copy()
    if "video_id" not in raw_flow_df.columns:
        raw_flow_df["video_id"] = "unknown"
    direction_tables = []
    for video_id, video_df in raw_flow_df.groupby("video_id", sort=False):
        meta = {c: video_df[c].dropna().iloc[0] for c in ["target_label", "target_category", "target_environment"] if c in video_df.columns and video_df[c].notna().any()}
        for camera in ["front", "rear"]:
            direction_df = build_flow_direction_jitter_score_table(video_df, camera)
            if len(direction_df):
                direction_df.insert(0, "video_id", video_id)
                for col, value in meta.items():
                    direction_df[col] = value
                direction_tables.append(direction_df)
    if not direction_tables:
        return pd.DataFrame(columns=empty_cols)

    broad_df = build_broad_vibration_score_table(pd.concat(direction_tables, ignore_index=True), top_k=top_k)
    if broad_df.empty:
        return pd.DataFrame(columns=empty_cols)

    broad_df = broad_df.copy()
    broad_df["time"] = broad_df["window_start_sec"].astype(float)
    broad_df["time_bin"] = np.round(broad_df["time"] / max(WINDOW_SEC, 1e-6)).astype(int)
    feature_df = broad_df.pivot_table(
        index=["video_id", "time_bin", "time"],
        columns="camera",
        values="broad_vibration_score",
        aggfunc="mean",
    ).reset_index().rename(columns={"front": "front_broad_vibration_score", "rear": "rear_broad_vibration_score"})
    feature_df.columns.name = None
    for col in BROAD_VIBRATION_COLUMNS:
        if col not in feature_df.columns:
            feature_df[col] = 0.0
    return feature_df[empty_cols].copy()


# 広域振動スコア列が欠けた場合でも、モデル入力の列が必ず存在するよう補完します。
# 関数メモ: ensure_broad_vibration_columns
# - 広域振動スコア列が欠けている場合に0で補完し、存在する場合は動画内で前後補完します。
# - 入力動画に片側カメラがない・計算に失敗した場合でも、モデル入力列を安定させます。

def ensure_broad_vibration_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in BROAD_VIBRATION_COLUMNS:
        if col not in df.columns:
            df[col] = 0.0
        else:
            values = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
            df[col] = values.groupby(df["video_id"]).transform(lambda s: s.ffill().bfill()).fillna(0.0) if "video_id" in df.columns else values.fillna(0.0)
    return df


# 関数メモ: extract_sample_features
# - 1サンプルの audio/front/rear を読み、音声特徴・raw flow・広域振動特徴を結合して返します。
# - 学習ノートでも推論ノートでも、1本分のモデル入力表を作る中心関数です。

def extract_sample_features(sample: pd.Series, return_raw_flow: bool = False):
    video_id = str(sample["sample_id"])
    feature_dfs = [extract_audio_features(sample["audio_path"], video_id=video_id)]
    raw_flow_dfs = []

    for prefix, path_col in [("front", "front_path"), ("rear", "rear_path")]:
        frames = extract_video_frames(sample[path_col])
        flow_df = compute_optical_flow_features(frames, prefix, video_id=video_id)
        feature_dfs.append(flow_df)
        raw_cols = [
            "video_id", "time", "time_bin",
            f"{prefix}_flow_x_mean", f"{prefix}_flow_y_mean",
            f"{prefix}_flow_x_std", f"{prefix}_flow_y_std",
            f"{prefix}_flow_mag_mean", f"{prefix}_flow_failed",
        ]
        for gy in range(FLOW_GRID[0]):
            for gx in range(FLOW_GRID[1]):
                raw_cols.extend([
                    f"{prefix}_flow_cell_{gy}_{gx}_x_mean",
                    f"{prefix}_flow_cell_{gy}_{gx}_y_mean",
                    f"{prefix}_flow_cell_{gy}_{gx}_valid_ratio",
                ])
        raw_cols = [c for c in raw_cols if c in flow_df.columns]
        if raw_cols:
            raw_flow_dfs.append(flow_df[raw_cols].copy())

    df = align_features_by_time(feature_dfs)

    if raw_flow_dfs:
        raw_flow_df = raw_flow_dfs[0]
        for other in raw_flow_dfs[1:]:
            raw_flow_df = raw_flow_df.merge(other.drop(columns=["time_bin"], errors="ignore"), on=["video_id", "time"], how="outer")
        raw_flow_df = raw_flow_df.sort_values(["video_id", "time"]).reset_index(drop=True)
        raw_flow_df["time_bin"] = np.round(raw_flow_df["time"] / RAW_FLOW_SAMPLE_SEC).astype(int)
        raw_flow_df["target_category"] = sample.get("category", "unknown")
        raw_flow_df["target_environment"] = sample.get("environment", "unknown")
        raw_flow_df["target_label"] = sample.get("plot_label", str(video_id))
    else:
        raw_flow_df = pd.DataFrame(columns=["video_id", "time", "time_bin", "target_category", "target_environment", "target_label"])

    broad_feature_df = build_broad_vibration_feature_df(raw_flow_df)
    if len(broad_feature_df):
        df = df.merge(broad_feature_df.drop(columns=["time"], errors="ignore"), on=["video_id", "time_bin"], how="left")
    df = ensure_broad_vibration_columns(df)
    df["environment"] = sample.get("environment", "unknown")
    df["label"] = sample.get("category", "unknown")

    if not return_raw_flow:
        return df
    return df, raw_flow_df


## 4. 推論と保存

In [6]:
# ------------------------------------------------------------
# セル概要: 特徴抽出・推論・保存
# ------------------------------------------------------------
# - 全ターゲットへ特徴量抽出を適用し、必要なら学習済みモデルで窓スコアを計算します。
# - ターゲット別CSVと全体比較CSVを保存し、後で横断比較できる形にします。
# ------------------------------------------------------------

# ============================================================
# 推論スコア計算
# ------------------------------------------------------------
# 推論で作った特徴量を、学習 artifact に保存されている feature_names の順番へ
# 並べ替えてから、audio / motion モデルで異常度を計算します。
# ============================================================
# 関数メモ: build_model_input
# - 推論特徴表を、学習時 artifact の feature_names と同じ列順・同じ前処理へ変換します。
# - 不足列は追加し、余分な列は落とし、モデルが期待する入力次元を厳密に再現します。

def build_model_input(inference_df: pd.DataFrame, feature_names: np.ndarray) -> pd.DataFrame:
        # パスやラベルなど、モデルへ入れてはいけないメタ情報を除外します。
    exclude_cols = {
        "time", "time_bin", "video_id", "sample_id", "label",
        "category", "target_category", "target_environment", "target_label", "plot_label",
        "front_path", "rear_path", "audio_path", "output_dir",
    }
    model_df = inference_df.drop(columns=[c for c in exclude_cols if c in inference_df.columns], errors="ignore")
    model_df = pd.get_dummies(model_df, columns=model_df.select_dtypes(include=["object", "category"]).columns, dummy_na=True)
        # 学習時の feature_names に reindex することで、列順と欠損列を安全にそろえます。
    return model_df.replace([np.inf, -np.inf], np.nan).reindex(columns=feature_names, fill_value=0.0)


# 関数メモ: apply_score_calibration
# - 保存済み分位点を使って raw score を0〜1へクリップ正規化します。
# - 学習時・推論時で同じキャリブレーションを使うことでスコアの意味をそろえます。

def apply_score_calibration(scores: np.ndarray | pd.Series, calibration: dict[str, float]) -> np.ndarray:
    values = np.asarray(scores, dtype=float)
    lower = float(calibration.get("lower", 0.0))
    upper = float(calibration.get("upper", lower + 1.0))
    denom = max(upper - lower, 1e-6)
    return np.clip((values - lower) / denom, 0.0, 1.0)


# 関数メモ: compute_temporal_sync_score
# - 音声異常度と動き異常度が近い時刻に同時に高いかを評価します。
# - 衝撃音だけ・動きだけより、両方が短時間内に出た窓を高く見積もります。

def compute_temporal_sync_score(
    scored_df: pd.DataFrame,
    audio_col: str = "audio_anomaly_score",
    motion_col: str = "motion_anomaly_score",
    max_lag_windows: int = 2,
) -> pd.Series:
    if audio_col not in scored_df.columns or motion_col not in scored_df.columns:
        return pd.Series(0.0, index=scored_df.index, name="sync_score")

    sync = pd.Series(0.0, index=scored_df.index, name="sync_score")
    sort_col = "time_bin" if "time_bin" in scored_df.columns else "time"
    window = max(1, int(max_lag_windows) * 2 + 1)

        # 音と動きが近いタイミングで高いとき、sync_score を高くします。
    for _, group in scored_df.sort_values(["video_id", sort_col]).groupby("video_id", sort=False):
        audio = group[audio_col].astype(float).fillna(0.0).clip(0.0, 1.0)
        motion = group[motion_col].astype(float).fillna(0.0).clip(0.0, 1.0)
        nearby_audio = audio.rolling(window=window, center=True, min_periods=1).max()
        nearby_motion = motion.rolling(window=window, center=True, min_periods=1).max()
        aligned = np.maximum(audio * nearby_motion, motion * nearby_audio)
        sync.loc[group.index] = np.sqrt(np.clip(aligned, 0.0, 1.0))
    return sync


# 関数メモ: add_composite_scores
# - audio / motion / sync の各スコアを重み付きで合成し、最終 anomaly_score を作ります。
# - 互換用に anomaly_score_smooth も追加し、グラフで見やすい移動平均を持たせます。

def add_composite_scores(
    scored_df: pd.DataFrame,
    sync_config: dict = sync_score_config,
    score_weights: dict[str, float] = final_score_weights,
) -> pd.DataFrame:
    scored_df = scored_df.copy()
    scored_df["sync_score"] = compute_temporal_sync_score(
        scored_df,
        audio_col=sync_config.get("audio_score_column", "audio_anomaly_score"),
        motion_col=sync_config.get("motion_score_column", "motion_anomaly_score"),
        max_lag_windows=int(sync_config.get("max_lag_windows", 2)),
    )

    available_weights = {col: float(weight) for col, weight in score_weights.items() if col in scored_df.columns and float(weight) > 0}
    weight_sum = sum(available_weights.values())
    if weight_sum <= 0:
        scored_df["final_anomaly_score"] = 0.0
    else:
        final_score = np.zeros(len(scored_df), dtype=float)
        for col, weight in available_weights.items():
            final_score += scored_df[col].astype(float).fillna(0.0).to_numpy() * (weight / weight_sum)
        scored_df["final_anomaly_score"] = np.clip(final_score, 0.0, 1.0)

    scored_df["anomaly_score"] = scored_df["final_anomaly_score"]
    scored_df["anomaly_score_smooth"] = scored_df.groupby("video_id")["anomaly_score"].transform(
        lambda s: s.rolling(window=5, center=True, min_periods=1).mean()
    )
    return scored_df


# 関数メモ: score_windows
# - 抽出済み特徴量へ各 score model を適用し、窓ごとの raw/calibrated score を追加します。
# - 最後に sync_score と anomaly_score を作り、ターゲット別の推論結果表を返します。

def score_windows(features_df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    scored_df = features_df.copy()
    model_inputs: dict[str, np.ndarray] = {}

        # audio / motion それぞれのモデルで raw score を出し、学習時の分位点で 0〜1 に変換します。
    for model_name, artifact in score_model_artifacts.items():
        feature_names = np.asarray(artifact["feature_names"])
        feature_weight_vector = np.asarray(artifact.get("feature_weight_vector", np.ones(len(feature_names))), dtype=float)
        X_scaled = artifact["preprocess_pipeline"].transform(build_model_input(features_df, feature_names))
        X = X_scaled * feature_weight_vector
        raw_scores = -artifact["model"].decision_function(X)
        score_col = artifact.get("score_column", f"{model_name}_anomaly_score")
        raw_score_col = artifact.get("raw_score_column", f"{model_name}_anomaly_score_raw")
        scored_df[raw_score_col] = raw_scores
        scored_df[score_col] = apply_score_calibration(raw_scores, artifact.get("score_calibration", {"lower": 0.0, "upper": 1.0}))
        model_inputs[model_name] = X

    scored_df = add_composite_scores(scored_df)
    return scored_df, model_inputs


target_results: dict[str, dict] = {}
all_feature_dfs = []
all_raw_flow_dfs = []
all_scored_dfs = []

# TARGET_SAMPLES を1本ずつ処理し、特徴量・raw flow・スコアCSVを保存します。
for sample in tqdm(TARGET_SAMPLES, desc="extract targets"):
    sample_id = str(sample["sample_id"])
    output_dir = Path(sample["output_dir"])
    features_df, raw_flow_df = extract_sample_features(sample, return_raw_flow=True)
    features_df = features_df.reset_index(drop=True)
    features_df["target_category"] = sample["category"]
    features_df["target_environment"] = sample["environment"]
    features_df["target_label"] = sample["plot_label"]

    features_path = output_dir / f"{sample_id}_features.csv"
    raw_flow_path = output_dir / f"{sample_id}_raw_flow_0p1s.csv"
    features_df.to_csv(features_path, index=False)
    raw_flow_df.to_csv(raw_flow_path, index=False)

    scored_df = None
    model_inputs: dict[str, np.ndarray] = {}
    if RUN_INFERENCE:
        scored_df, model_inputs = score_windows(features_df)
        scored_df = scored_df.reset_index(drop=True)
        scored_df["target_category"] = sample["category"]
        scored_df["target_environment"] = sample["environment"]
        scored_df["target_label"] = sample["plot_label"]
        scores_path = output_dir / f"{sample_id}_window_scores.csv"
        scored_df.to_csv(scores_path, index=False)
        all_scored_dfs.append(scored_df)
        print(f"{sample_id}: {len(scored_df)} scored windows -> {scores_path}")
    else:
        print(f"{sample_id}: inference skipped, features={len(features_df)} windows, raw_flow={len(raw_flow_df)} rows -> {raw_flow_path}")

    target_results[sample_id] = {
        "sample": sample,
        "features_df": features_df,
        "raw_flow_df": raw_flow_df,
        "scored_df": scored_df,
        "model_inputs": model_inputs,
    }
    all_feature_dfs.append(features_df)
    all_raw_flow_dfs.append(raw_flow_df)

all_target_features_df = pd.concat(all_feature_dfs, ignore_index=True) if all_feature_dfs else pd.DataFrame()
all_target_raw_flow_df = pd.concat(all_raw_flow_dfs, ignore_index=True) if all_raw_flow_dfs else pd.DataFrame()
all_target_features_path = TARGET_COMPARISON_DIR / "all_target_features.csv"
all_target_raw_flow_path = TARGET_COMPARISON_DIR / "all_target_raw_flow_0p1s.csv"
all_target_features_df.to_csv(all_target_features_path, index=False)
all_target_raw_flow_df.to_csv(all_target_raw_flow_path, index=False)
print(f"combined features: {all_target_features_df.shape} -> {all_target_features_path}")
print(f"combined raw flow: {all_target_raw_flow_df.shape} -> {all_target_raw_flow_path}")

default_display_cols = ["video_id", "target_label", "time", "front_flow_x_mean", "front_flow_y_mean", "rear_flow_x_mean", "rear_flow_y_mean"]
display_cols = [c for c in default_display_cols if c in all_target_raw_flow_df.columns]
if display_cols:
    display(all_target_raw_flow_df[display_cols].head(12))

if RUN_INFERENCE and all_scored_dfs:
    all_target_scored_df = pd.concat(all_scored_dfs, ignore_index=True)
    all_target_scores_path = TARGET_COMPARISON_DIR / "all_target_window_scores.csv"
    all_target_scored_df.to_csv(all_target_scores_path, index=False)
    print(f"combined scores: {all_target_scored_df.shape} -> {all_target_scores_path}")
    score_display_cols = ["video_id", "target_label", "time", *[c for c in PLOT_SCORE_COLUMNS if c in all_target_scored_df.columns], "anomaly_score_smooth"]
    display(all_target_scored_df[[c for c in score_display_cols if c in all_target_scored_df.columns]].head(12))
else:
    all_target_scored_df = pd.DataFrame()


extract targets:   0%|          | 0/50 [00:00<?, ?it/s]

084_旋回時、柱に衝突: 76 scored windows -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/084_旋回時、柱に衝突/084_旋回時、柱に衝突_window_scores.csv
1030: 73 scored windows -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/1030/1030_window_scores.csv
1033: 80 scored windows -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/1033/1033_window_scores.csv
1017: 74 scored windows -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/1017/1017_window_scores.csv
043_ラック下を通過時マストが衝突: 73 scored windows -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/043_ラック下を通過時マストが衝突/043_ラック下を通過時マストが衝突_window_scores.csv
1015: 77 scored windows -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/1015/1015_window_scores.csv
1063: 78 scored windows -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/1063/1063_window_scores.csv
1062: 80 scored windows -> /workspaces/forklift/outputs/anomaly_feature_poc/target_infe

,video_id,target_label,time,front_flow_x_mean,front_flow_y_mean,rear_flow_x_mean,rear_flow_y_mean
0,084_旋回時、柱に衝突,anomaly/indoor/084,0.099933,0.750300,0.499821,0.003560,0.002806
1,084_旋回時、柱に衝突,anomaly/indoor/084,0.199867,-0.000142,0.000392,0.000217,-0.000475
2,084_旋回時、柱に衝突,anomaly/indoor/084,0.299800,1.083891,-0.351995,-0.000526,-0.000702
3,084_旋回時、柱に衝突,anomaly/indoor/084,0.399734,-0.614531,-2.288283,0.003510,-0.002972
4,084_旋回時、柱に衝突,anomaly/indoor/084,0.499667,-0.560994,-2.068566,-0.115560,1.865670
5,084_旋回時、柱に衝突,anomaly/indoor/084,0.599600,0.324523,-1.157394,-0.000995,0.002197
6,084_旋回時、柱に衝突,anomaly/indoor/084,0.699534,0.598079,-0.903152,-1.277943,3.387907
7,084_旋回時、柱に衝突,anomaly/indoor/084,0.799467,-0.217973,-1.888766,-0.000956,-0.000207
8,084_旋回時、柱に衝突,anomaly/indoor/084,0.899400,-0.162063,-1.952486,-1.673360,3.645956
9,084_旋回時、柱に衝突,anomaly/indoor/084,0.999334,-0.322464,-2.089174,-0.000561,0.002034


combined scores: (3762, 153) -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/all_target_window_scores.csv


,video_id,target_label,time,anomaly_score,audio_anomaly_score,motion_anomaly_score,sync_score,anomaly_score_smooth
0,084_旋回時、柱に衝突,anomaly/indoor/084,0.0,0.121295,0.083907,0.118671,0.210008,0.223271
1,084_旋回時、柱に衝突,anomaly/indoor/084,0.2,0.219733,0.273469,0.118671,0.275683,0.233515
2,084_旋回時、柱に衝突,anomaly/indoor/084,0.4,0.328787,0.371646,0.277914,0.321381,0.239630
3,084_旋回時、柱に衝突,anomaly/indoor/084,0.6,0.264247,0.228224,0.277914,0.321381,0.266511
4,084_旋回時、柱に衝突,anomaly/indoor/084,0.8,0.264090,0.227874,0.277914,0.321381,0.277912
5,084_旋回時、柱に衝突,anomaly/indoor/084,1.0,0.255698,0.260369,0.242092,0.268998,0.266278
6,084_旋回時、柱に衝突,anomaly/indoor/084,1.2,0.276741,0.298644,0.242092,0.288093,0.245848
7,084_旋回時、柱に衝突,anomaly/indoor/084,1.4,0.270616,0.293571,0.242092,0.268885,0.233371
8,084_旋回時、柱に衝突,anomaly/indoor/084,1.6,0.162096,0.171367,0.126413,0.203683,0.216177
9,084_旋回時、柱に衝突,anomaly/indoor/084,1.8,0.201706,0.242278,0.126413,0.242185,0.179058


## 5. 比較グラフと上位窓

`PLOT_SCORE_COLUMNS` でスコア系、`PLOT_FEATURE_COLUMNS` で特徴量系の表示列を切り替えます。`anomaly_score` は合成後の最終異常スコアです。

In [7]:
# ------------------------------------------------------------
# セル概要: 比較グラフと特徴量可視化
# ------------------------------------------------------------
# - 異常スコア、代表特徴、raw flow、方向変化、広域振動スコアなどを図として保存します。
# - スコアが高い窓について、音声由来か動き由来かを切り分けるための診断用セルです。
# ------------------------------------------------------------

# ============================================================
# 推論結果と raw flow の可視化
# ------------------------------------------------------------
# 異常スコア、広域振動スコア、raw flow の変化量を画像として保存します。
# raw flow 系の図は、なぜ motion score が上がったかを確認する診断用です。
# ============================================================
# 関数メモ: plot_anomaly_scores
# - 窓ごとの anomaly/audio/motion/sync スコアを時系列グラフとして描きます。
# - ピークがどの時刻にあり、どのスコア成分が高いかを素早く確認します。

def plot_anomaly_scores(scored_df: pd.DataFrame, save_path: str | Path | None = None):
    label = scored_df["target_label"].iloc[0]
    feature_plot_cols = PLOT_FEATURE_COLUMNS if PLOT_FEATURE_GRAPHS else []
    cols = [c for c in [*PLOT_SCORE_COLUMNS, *feature_plot_cols] if c in scored_df.columns]
    if not cols:
        print("No anomaly plot columns found.")
        return None
    axes = scored_df.plot(
        x="time",
        y=cols,
        subplots=True,
        figsize=(14, 2.2 * len(cols)),
        legend=True,
        title=f"target={label}",
    )
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    return axes


def plot_overlay_metric(
    scored_df: pd.DataFrame,
    metric_col: str,
    ax=None,
):
    if ax is None:
        _, ax = plt.subplots(figsize=(14, 4))
    for _video_id, df in scored_df.groupby("video_id", sort=False):
        if metric_col not in df.columns:
            continue
        df = df.sort_values("time")
        ax.plot(df["time"], df[metric_col], linewidth=1.6, label=df["target_label"].iloc[0])
    ax.set_title(metric_col)
    ax.set_xlabel("time [sec]")
    ax.set_ylabel(metric_col)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    return ax


# 関数メモ: plot_overlay_metrics
# - 複数の特徴量を同じ時間軸で重ねて描きます。
# - 音声特徴と広域振動スコアなど、異なる指標のピーク位置を比較します。

def plot_overlay_metrics(
    scored_df: pd.DataFrame,
    metric_cols: list[str],
    save_path: str | Path | None = None,
):
    metric_cols = [c for c in metric_cols if c in scored_df.columns]
    if not metric_cols:
        print("No overlay metric columns found.")
        return None
    fig, axes = plt.subplots(len(metric_cols), 1, figsize=(14, 3.0 * len(metric_cols)), sharex=False)
    if len(metric_cols) == 1:
        axes = [axes]
    for ax, metric_col in zip(axes, metric_cols):
        plot_overlay_metric(scored_df, metric_col, ax=ax)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    return axes


# 3x3 グリッドごとの flow x/y をそのまま描き、入力信号の形を確認します。
# 関数メモ: plot_raw_flow_xy
# - raw flow の x/y 平均を camera 別に描画します。
# - モデル入力前の動きそのものがどの向きに変化しているかを確認します。

def plot_raw_flow_xy(raw_flow_df: pd.DataFrame, camera: str, save_path: str | Path | None = None):
    """Plot per-grid raw flow x/y as a 3x3 camera map."""
    if raw_flow_df.empty:
        print(f"No raw flow rows found for {camera}.")
        return None

    grid_rows, grid_cols = FLOW_GRID
    label = raw_flow_df["target_label"].iloc[0] if "target_label" in raw_flow_df.columns and len(raw_flow_df) else "raw_flow"
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
            y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
            valid_col = f"{camera}_flow_cell_{gy}_{gx}_valid_ratio"
            if x_col in raw_flow_df.columns:
                ax.plot(raw_flow_df["time"], raw_flow_df[x_col], linewidth=1.0, label="x")
                plotted = True
            if y_col in raw_flow_df.columns:
                ax.plot(raw_flow_df["time"], raw_flow_df[y_col], linewidth=1.0, label="y")
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35)
            title = f"{gx + 1}x{gy + 1}_{camera} mag>={FLOW_DIRECTION_MIN_MAG:g}"
            if valid_col in raw_flow_df.columns:
                valid_mean = raw_flow_df[valid_col].astype(float).mean()
                title += f" valid={valid_mean:.2f}"
            ax.set_title(title, fontsize=10)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel("flow mean")
            if gy == grid_rows - 1:
                ax.set_xlabel("time [sec]")
            ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"raw optical flow {camera} grid x/y 0.1s: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} per-grid raw flow x/y columns found.")
    return axes


# 関数メモ: add_flow_window_score_alpha
# - flow component の窓スコアへ可視化用 alpha を付与します。
# - 選択窓だけを濃く表示できるよう、スコア範囲を透明度へ変換します。

def add_flow_window_score_alpha(window_df: pd.DataFrame) -> pd.DataFrame:
    window_df = window_df.copy()
    if window_df.empty or "flow_window_score" not in window_df.columns:
        window_df["flow_window_score_alpha"] = 0.0
        return window_df
    scores = window_df["flow_window_score"].to_numpy(dtype=float)
    finite = scores[np.isfinite(scores)]
    if finite.size == 0 or float(np.nanmax(finite)) <= float(FLOW_SCORE_MIN_VISIBLE):
        window_df["flow_window_score_alpha"] = 0.0
        return window_df
    lower = float(np.nanmin(finite))
    upper = float(np.nanmax(finite))
    if upper <= lower:
        normalized = np.where(scores > FLOW_SCORE_MIN_VISIBLE, 1.0, 0.0)
    else:
        normalized = np.clip((np.nan_to_num(scores, nan=lower) - lower) / max(upper - lower, 1e-6), 0.0, 1.0)
    alpha = FLOW_SCORE_ALPHA_MIN + normalized * (FLOW_SCORE_ALPHA_MAX - FLOW_SCORE_ALPHA_MIN)
    alpha = np.where(scores > FLOW_SCORE_MIN_VISIBLE, alpha, 0.0)
    window_df["flow_window_score_alpha"] = alpha.astype(float)
    return window_df


# 関数メモ: compute_flow_component_change_abs
# - 1つの flow 成分について、連続時刻の絶対変化量を計算します。
# - x/y/magnitude などの急変を窓スコア化する前処理です。

def compute_flow_component_change_abs(raw_flow_df: pd.DataFrame, col: str) -> pd.Series:
    """Compute |d(flow component)/dt| from raw per-grid flow values."""
    if col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.Series(dtype=float)
    ordered = raw_flow_df.sort_values("time")
    time = ordered["time"].astype(float)
    dt = time.diff().replace([np.inf, -np.inf], np.nan).fillna(RAW_FLOW_SAMPLE_SEC).clip(lower=1e-6)
    change_abs = (ordered[col].astype(float).diff().fillna(0.0) / dt).abs()
    return pd.Series(change_abs.to_numpy(dtype=float), index=ordered.index).reindex(raw_flow_df.index).fillna(0.0)


# flow ベクトルそのものの変化量 sqrt((dx/dt)^2 + (dy/dt)^2) を計算します。
# 関数メモ: compute_flow_vector_change_abs
# - flow x/y ベクトルの連続時刻差分の大きさを計算します。
# - 向きと大きさを合わせた移動ベクトル変化を見たい場合に使います。

def compute_flow_vector_change_abs(raw_flow_df: pd.DataFrame, x_col: str, y_col: str) -> pd.Series:
    """Compute sqrt((dx/dt)^2 + (dy/dt)^2) from raw per-grid flow vectors."""
    if x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.Series(dtype=float)
    ordered = raw_flow_df.sort_values("time")
    time = ordered["time"].astype(float)
    dt = time.diff().replace([np.inf, -np.inf], np.nan).fillna(RAW_FLOW_SAMPLE_SEC).clip(lower=1e-6)
    dx_dt = ordered[x_col].astype(float).diff().fillna(0.0) / dt
    dy_dt = ordered[y_col].astype(float).diff().fillna(0.0) / dt
    vector_change_abs = np.hypot(dx_dt.to_numpy(dtype=float), dy_dt.to_numpy(dtype=float))
    vector_change_abs = np.nan_to_num(vector_change_abs, nan=0.0, posinf=0.0, neginf=0.0)
    return pd.Series(vector_change_abs, index=ordered.index).reindex(raw_flow_df.index).fillna(0.0)


# flow の大きさだけに注目し、abs(d|flow|/dt) を計算します。
# 関数メモ: compute_flow_magnitude_change_abs
# - flow ベクトルの magnitude の時間変化量を計算します。
# - 動きの方向ではなく、動きの強さが急に変わったかを評価します。

def compute_flow_magnitude_change_abs(raw_flow_df: pd.DataFrame, x_col: str, y_col: str) -> pd.Series:
    """Compute abs(d(sqrt(x^2 + y^2))/dt) from raw per-grid flow vectors."""
    if x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.Series(dtype=float)
    ordered = raw_flow_df.sort_values("time")
    time = ordered["time"].astype(float)
    dt = time.diff().replace([np.inf, -np.inf], np.nan).fillna(RAW_FLOW_SAMPLE_SEC).clip(lower=1e-6)
    magnitude = np.hypot(ordered[x_col].astype(float).to_numpy(), ordered[y_col].astype(float).to_numpy())
    magnitude_change_abs = np.abs(pd.Series(magnitude, index=ordered.index).diff().fillna(0.0).to_numpy(dtype=float) / dt.to_numpy(dtype=float))
    magnitude_change_abs = np.nan_to_num(magnitude_change_abs, nan=0.0, posinf=0.0, neginf=0.0)
    return pd.Series(magnitude_change_abs, index=ordered.index).reindex(raw_flow_df.index).fillna(0.0)


# 大きさ変化グラフを見やすくするため、短周期ノイズをローパスでならします。
# 関数メモ: lowpass_raw_flow_series
# - raw flow 系列へローパスフィルタをかけ、ゆっくりした成分を取り出します。
# - 細かいノイズを抑え、広域の揺れや傾向を見やすくするための診断用処理です。

def lowpass_raw_flow_series(
    values: pd.Series | np.ndarray,
    sample_sec: float = RAW_FLOW_SAMPLE_SEC,
    cutoff_hz: float = RAW_FLOW_MAGNITUDE_LOWPASS_CUTOFF_HZ,
    order: int = RAW_FLOW_MAGNITUDE_LOWPASS_ORDER,
) -> np.ndarray:
    """Low-pass filter a raw flow time series, falling back to a centered mean for short signals."""
    values_arr = np.asarray(values, dtype=float)
    values_arr = np.nan_to_num(values_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if values_arr.size < 3:
        return values_arr
    sample_rate_hz = 1.0 / max(float(sample_sec), 1e-6)
    nyquist_hz = 0.5 * sample_rate_hz
    normalized_cutoff = float(cutoff_hz) / max(nyquist_hz, 1e-6)
    normalized_cutoff = float(np.clip(normalized_cutoff, 1e-4, 0.999))
    filter_order = max(1, int(order))
    b, a = butter(filter_order, normalized_cutoff, btype="low")
    padlen = 3 * max(len(a), len(b))
    if values_arr.size <= padlen:
        window = max(3, min(values_arr.size, int(round(sample_rate_hz / max(float(cutoff_hz), 1e-6)))))
        if window % 2 == 0 and window > 1:
            window -= 1
        return pd.Series(values_arr).rolling(window=window, center=True, min_periods=1).mean().to_numpy(dtype=float)
    return filtfilt(b, a, values_arr)



# 関数メモ: build_flow_component_window_table
# - flow 成分の変化量を固定長窓へ集約し、窓スコア表を作ります。
# - raw flow のどのグリッド・成分が異常候補時間に反応したかを確認します。

def build_flow_component_window_table(
    raw_flow_df: pd.DataFrame,
    camera: str,
    component: str,
    gy: int,
    gx: int,
    window_sec: float = FLOW_SCORE_WINDOW_SEC,
    hop_sec: float = FLOW_SCORE_HOP_SEC,
) -> pd.DataFrame:
    value_col = f"{camera}_flow_cell_{gy}_{gx}_{component}_mean"
    if raw_flow_df.empty or value_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.DataFrame()

    ordered = raw_flow_df.sort_values("time").reset_index(drop=True)
    change_abs = compute_flow_component_change_abs(ordered, value_col).to_numpy(dtype=float)
    times = ordered["time"].astype(float).to_numpy()
    if times.size == 0:
        return pd.DataFrame()
    duration_sec = float(np.nanmax(times)) if np.isfinite(times).any() else 0.0
    duration_sec = max(duration_sec, float(window_sec))

    rows = []
    for start_sec in build_flow_window_starts(duration_sec=duration_sec, window_sec=window_sec, hop_sec=hop_sec):
        end_sec = float(min(start_sec + window_sec, duration_sec))
        start_sec = max(end_sec - window_sec, 0.0)
        mask = (times >= start_sec) & (times < end_sec)
        if not np.any(mask):
            nearest_index = int(np.argmin(np.abs(times - start_sec)))
            mask = np.zeros_like(times, dtype=bool)
            mask[nearest_index] = True
        segment = change_abs[mask]
        change_mean = float(np.mean(segment))
        change_p95 = float(np.percentile(segment, 95))
        change_max = float(np.max(segment))
        if change_max > FLOW_SCORE_MIN_VISIBLE:
            high_threshold = change_max * float(FLOW_SCORE_HIGH_RATIO_FRACTION)
            high_ratio = float(np.mean(segment >= high_threshold))
            peakiness = float(np.clip((change_max - change_mean) / max(change_max, 1e-6), 0.0, 1.0))
            distinctiveness = float(np.clip(peakiness * np.sqrt(max(0.0, 1.0 - high_ratio)), 0.0, 1.0))
        else:
            high_ratio = 0.0
            peakiness = 0.0
            distinctiveness = 0.0
        rows.append({
            "camera": camera,
            "component": component,
            "grid_x": int(gx + 1),
            "grid_y": int(gy + 1),
            "grid_col": int(gx),
            "grid_row": int(gy),
            "window_start_sec": float(start_sec),
            "window_end_sec": float(end_sec),
            "window_center_sec": float(0.5 * (start_sec + end_sec)),
            "change_mean": change_mean,
            "change_p95": change_p95,
            "change_max": change_max,
            "change_high_ratio": high_ratio,
            "change_peakiness": peakiness,
            "flow_window_distinctiveness": distinctiveness,
        })

    window_df = pd.DataFrame(rows)
    for feature_name in ["change_mean", "change_p95", "change_max"]:
        window_df[f"{feature_name}_z"] = robust_positive_zscore(window_df[feature_name].to_numpy(dtype=float))
    base_intensity_score = (
        0.20 * window_df["change_mean_z"]
        + 0.50 * window_df["change_p95_z"]
        + 0.30 * window_df["change_max_z"]
    ).astype(float)
    distinctiveness_multiplier = (
        (1.0 - float(FLOW_SCORE_DISTINCTIVENESS_WEIGHT))
        + float(FLOW_SCORE_DISTINCTIVENESS_WEIGHT) * window_df["flow_window_distinctiveness"].astype(float)
    )
    window_df["flow_window_base_intensity_score"] = base_intensity_score
    window_df["flow_window_score"] = (base_intensity_score * distinctiveness_multiplier).astype(float)
    return add_flow_window_score_alpha(window_df)


# 関数メモ: build_flow_grid_window_score_table
# - camera と component を指定して、全グリッドの flow 窓スコア表を作ります。
# - グリッド別の動き変化を比較し、可視化用の土台にします。

def build_flow_grid_window_score_table(raw_flow_df: pd.DataFrame, camera: str, component: str) -> pd.DataFrame:
    rows = []
    for gy in range(FLOW_GRID[0]):
        for gx in range(FLOW_GRID[1]):
            window_df = build_flow_component_window_table(raw_flow_df, camera, component, gy, gx)
            if len(window_df):
                rows.append(window_df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


# 関数メモ: plot_raw_flow_component_change_abs
# - flow 成分変化量と窓スコアをグリッド別に描画します。
# - どの領域の x/y 変化が強かったかを診断するための図を保存します。

def plot_raw_flow_component_change_abs(
    raw_flow_df: pd.DataFrame,
    camera: str,
    component: str,
    save_path: str | Path | None = None,
    window_score_df: pd.DataFrame | None = None,
):
    """Plot per-grid absolute x or y flow change as a 3x3 camera map."""
    if raw_flow_df.empty:
        print(f"No raw flow rows found for {camera}.")
        return None
    if component not in {"x", "y"}:
        raise ValueError("component must be 'x' or 'y'")

    grid_rows, grid_cols = FLOW_GRID
    label = raw_flow_df["target_label"].iloc[0] if "target_label" in raw_flow_df.columns and len(raw_flow_df) else "raw_flow"
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            value_col = f"{camera}_flow_cell_{gy}_{gx}_{component}_mean"
            valid_col = f"{camera}_flow_cell_{gy}_{gx}_valid_ratio"
            change_abs = compute_flow_component_change_abs(raw_flow_df, value_col)
            scored_windows = pd.DataFrame()
            if window_score_df is not None and not window_score_df.empty:
                scored_windows = window_score_df[
                    window_score_df["camera"].astype(str).eq(camera)
                    & window_score_df["component"].astype(str).eq(component)
                    & window_score_df["grid_row"].astype(int).eq(gy)
                    & window_score_df["grid_col"].astype(int).eq(gx)
                ]
                for window_row in scored_windows.itertuples(index=False):
                    alpha = float(getattr(window_row, "flow_window_score_alpha", 0.0))
                    if alpha <= 0.0:
                        continue
                    ax.axvspan(
                        float(window_row.window_start_sec),
                        float(window_row.window_end_sec),
                        color="tab:orange",
                        alpha=alpha,
                        zorder=0,
                    )
            if len(change_abs):
                ax.plot(raw_flow_df["time"], change_abs, linewidth=1.0, label=f"|d{component}/dt|", zorder=2)
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35, zorder=1)
            title = f"{gx + 1}x{gy + 1}_{camera} mag>={FLOW_DIRECTION_MIN_MAG:g}"
            if valid_col in raw_flow_df.columns:
                valid_mean = raw_flow_df[valid_col].astype(float).mean()
                title += f" valid={valid_mean:.2f}"
            if len(scored_windows):
                title += f" max={scored_windows['flow_window_score'].max():.2f}"
            ax.set_title(title, fontsize=10)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel(f"|d{component}/dt|")
            if gy == grid_rows - 1:
                ax.set_xlabel("time [sec]")
            ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"raw optical flow {camera} grid absolute {component}-change 0.1s: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} per-grid raw flow {component} columns found for absolute change.")
    return axes


# 関数メモ: plot_raw_flow_vector_change_abs
# - flow ベクトル変化量を camera 別に描画します。
# - 方向と大きさを合わせた動きの急変をグリッド単位で見ます。

def plot_raw_flow_vector_change_abs(raw_flow_df: pd.DataFrame, camera: str, save_path: str | Path | None = None):
    """Plot per-grid raw flow vector change magnitude as a 3x3 camera map."""
    if raw_flow_df.empty:
        print(f"No raw flow rows found for {camera}.")
        return None

    grid_rows, grid_cols = FLOW_GRID
    label = raw_flow_df["target_label"].iloc[0] if "target_label" in raw_flow_df.columns and len(raw_flow_df) else "raw_flow"
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
            y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
            valid_col = f"{camera}_flow_cell_{gy}_{gx}_valid_ratio"
            vector_change_abs = compute_flow_vector_change_abs(raw_flow_df, x_col, y_col)
            if len(vector_change_abs):
                ax.plot(raw_flow_df["time"], vector_change_abs, linewidth=1.0, label="|dflow/dt|", zorder=2)
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35, zorder=1)
            title = f"{gx + 1}x{gy + 1}_{camera}"
            if valid_col in raw_flow_df.columns:
                valid_mean = raw_flow_df[valid_col].astype(float).mean()
                title += f" valid={valid_mean:.2f}"
            ax.set_title(title, fontsize=10)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel("|dflow/dt|")
            if gy == grid_rows - 1:
                ax.set_xlabel("time [sec]")
            ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"raw optical flow {camera} grid vector change 0.1s: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} per-grid raw flow x/y columns found for vector change.")
    return axes


def plot_raw_flow_magnitude_change_abs(raw_flow_df: pd.DataFrame, camera: str, save_path: str | Path | None = None):
    """Plot per-grid raw flow magnitude change as a 3x3 camera map."""
    if raw_flow_df.empty:
        print(f"No raw flow rows found for {camera}.")
        return None

    grid_rows, grid_cols = FLOW_GRID
    label = raw_flow_df["target_label"].iloc[0] if "target_label" in raw_flow_df.columns and len(raw_flow_df) else "raw_flow"
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
            y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
            valid_col = f"{camera}_flow_cell_{gy}_{gx}_valid_ratio"
            magnitude_change_abs = compute_flow_magnitude_change_abs(raw_flow_df, x_col, y_col)
            if len(magnitude_change_abs):
                ax.plot(raw_flow_df["time"], magnitude_change_abs, linewidth=1.0, label="|d|flow|/dt|", zorder=2)
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35, zorder=1)
            title = f"{gx + 1}x{gy + 1}_{camera}"
            if valid_col in raw_flow_df.columns:
                valid_mean = raw_flow_df[valid_col].astype(float).mean()
                title += f" valid={valid_mean:.2f}"
            ax.set_title(title, fontsize=10)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel("|d|flow|/dt|")
            if gy == grid_rows - 1:
                ax.set_xlabel("time [sec]")
            ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"raw optical flow {camera} grid magnitude change 0.1s: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} per-grid raw flow x/y columns found for magnitude change.")
    return axes


# 関数メモ: plot_raw_flow_magnitude_change_abs_lowpass
# - ローパス後の flow magnitude 変化を描画します。
# - 細かい揺れよりも広域・低周波の変化を見たい場合の診断図です。

def plot_raw_flow_magnitude_change_abs_lowpass(raw_flow_df: pd.DataFrame, camera: str, save_path: str | Path | None = None):
    """Plot low-pass filtered per-grid raw flow magnitude change as a 3x3 camera map."""
    if raw_flow_df.empty:
        print(f"No raw flow rows found for {camera}.")
        return None

    grid_rows, grid_cols = FLOW_GRID
    label = raw_flow_df["target_label"].iloc[0] if "target_label" in raw_flow_df.columns and len(raw_flow_df) else "raw_flow"
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
            y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
            valid_col = f"{camera}_flow_cell_{gy}_{gx}_valid_ratio"
            magnitude_change_abs = compute_flow_magnitude_change_abs(raw_flow_df, x_col, y_col)
            if len(magnitude_change_abs):
                filtered = lowpass_raw_flow_series(magnitude_change_abs)
                ax.plot(raw_flow_df["time"], filtered, linewidth=1.2, label="lowpass |d|flow|/dt|", zorder=2)
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35, zorder=1)
            title = f"{gx + 1}x{gy + 1}_{camera} lp<={RAW_FLOW_MAGNITUDE_LOWPASS_CUTOFF_HZ:g}Hz"
            if valid_col in raw_flow_df.columns:
                valid_mean = raw_flow_df[valid_col].astype(float).mean()
                title += f" valid={valid_mean:.2f}"
            ax.set_title(title, fontsize=10)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel("lowpass |d|flow|/dt|")
            if gy == grid_rows - 1:
                ax.set_xlabel("time [sec]")
            ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"low-pass raw optical flow {camera} grid magnitude change 0.1s: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} per-grid raw flow x/y columns found for low-pass magnitude change.")
    return axes


# 振動スコアの元になる向き変化 |dtheta| と、選択された窓を重ねて描きます。
# 関数メモ: plot_raw_flow_direction_change
# - direction jitter の元になる向き変化量と選択窓を描画します。
# - 広域振動スコアが高い理由を、グリッド別の向き変化として確認します。

def plot_raw_flow_direction_change(
    raw_flow_df: pd.DataFrame,
    camera: str,
    save_path: str | Path | None = None,
    direction_score_df: pd.DataFrame | None = None,
):
    """Plot per-grid wrapped flow direction change as a 3x3 camera map."""
    if raw_flow_df.empty:
        print(f"No raw flow rows found for {camera}.")
        return None

    grid_rows, grid_cols = FLOW_GRID
    label = raw_flow_df["target_label"].iloc[0] if "target_label" in raw_flow_df.columns and len(raw_flow_df) else "raw_flow"
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
            y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
            valid_col = f"{camera}_flow_cell_{gy}_{gx}_valid_ratio"
            scored_windows = pd.DataFrame()
            if direction_score_df is not None and len(direction_score_df):
                scored_windows = direction_score_df[
                    direction_score_df["camera"].astype(str).eq(camera)
                    & direction_score_df["grid_row"].astype(int).eq(gy)
                    & direction_score_df["grid_col"].astype(int).eq(gx)
                ]
                for window_row in scored_windows.itertuples(index=False):
                    alpha = float(getattr(window_row, "direction_jitter_score_alpha", 0.0))
                    if alpha <= 0.0:
                        continue
                    ax.axvspan(
                        float(window_row.window_start_sec),
                        float(window_row.window_end_sec),
                        color="tab:orange",
                        alpha=alpha,
                        zorder=0,
                    )
            direction_change = compute_flow_direction_change_abs(raw_flow_df, x_col, y_col)
            if len(direction_change):
                ax.plot(raw_flow_df["time"], direction_change, linewidth=1.0, label="|dtheta|", zorder=2)
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35, zorder=1)
            ax.axhline(np.pi, color="tab:red", linewidth=0.7, alpha=0.25, linestyle="--", zorder=1)
            title = f"{gx + 1}x{gy + 1}_{camera} mag>={FLOW_DIRECTION_MIN_MAG:g}"
            if valid_col in raw_flow_df.columns:
                valid_mean = raw_flow_df[valid_col].astype(float).mean()
                title += f" valid={valid_mean:.2f}"
            if len(scored_windows):
                selected_count = int(scored_windows.get("direction_jitter_selected", pd.Series(dtype=bool)).sum())
                title += f" max={scored_windows['direction_jitter_score'].max():.2f} sel={selected_count}"
            ax.set_title(title, fontsize=10)
            ax.set_ylim(0.0, np.pi)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel("|dtheta| [rad]")
            if gy == grid_rows - 1:
                ax.set_xlabel("time [sec]")
            ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"raw optical flow {camera} grid direction change 0.1s: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} per-grid raw flow x/y columns found for direction change.")
    return axes


# front/rear 別の広域振動スコアを時系列で描きます。
# 関数メモ: plot_broad_vibration_score
# - front/rear の広域振動スコアを時系列で描きます。
# - motion モデルが実際に見ている代表特徴のピーク位置を確認します。

def plot_broad_vibration_score(broad_vibration_df: pd.DataFrame, save_path: str | Path | None = None, title: str | None = None):
    if broad_vibration_df is None or broad_vibration_df.empty:
        print("No broad vibration score rows found.")
        return None

    if "camera" in broad_vibration_df.columns:
        camera_values = [c for c in ["front", "rear"] if c in set(broad_vibration_df["camera"].astype(str))]
        camera_values.extend(sorted(set(broad_vibration_df["camera"].astype(str)) - set(camera_values)))
    else:
        camera_values = [None]

    fig, axes = plt.subplots(len(camera_values), 1, figsize=(14, 3.6 * len(camera_values)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, camera in zip(axes, camera_values):
        if camera is None:
            camera_df = broad_vibration_df.copy()
            group_cols = ["video_id"]
            subplot_title = title or f"broad vibration score top{BROAD_VIBRATION_TOP_K}"
        else:
            camera_df = broad_vibration_df[broad_vibration_df["camera"].astype(str).eq(camera)].copy()
            group_cols = ["video_id"]
            subplot_title = f"{camera}"
        for _video_id, df in camera_df.sort_values(["video_id", "window_start_sec"]).groupby(group_cols, sort=False):
            label = df["target_label"].iloc[0] if "target_label" in df.columns else str(_video_id)
            ax.plot(df["window_start_sec"], df["broad_vibration_score"], marker="o", markersize=3, linewidth=1.5, label=label)
        ax.set_title(subplot_title)
        ax.set_ylabel("broad vibration score")
        ax.set_ylim(0.0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best", fontsize=8)
    axes[-1].set_xlabel("window start [sec]")
    if title and len(camera_values) > 1:
        fig.suptitle(title, y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    return ax


# グリッドごとの振動スコアだけを描き、同時刻 top-k のグリッドに色を付けます。
# 関数メモ: plot_direction_jitter_score_grid
# - 3x3 グリッドの direction jitter score を小分けグラフで表示します。
# - どのグリッドが top-k に入りやすいか、局所振動の分布を確認します。

def plot_direction_jitter_score_grid(direction_jitter_df: pd.DataFrame, camera: str, save_path: str | Path | None = None):
    """Plot only per-window vibration scores and mark top-k grids for each timestamp."""
    if direction_jitter_df is None or direction_jitter_df.empty:
        print(f"No direction jitter score rows found for {camera}.")
        return None

    camera_df = direction_jitter_df[direction_jitter_df["camera"].astype(str).eq(camera)].copy()
    if camera_df.empty:
        print(f"No direction jitter score rows found for {camera}.")
        return None

    if "thresholded_direction_jitter_score" not in camera_df.columns or "direction_jitter_topk_rank" not in camera_df.columns:
        camera_df = add_direction_jitter_topk_columns(camera_df)

    grid_rows, grid_cols = FLOW_GRID
    label = camera_df["target_label"].iloc[0] if "target_label" in camera_df.columns and len(camera_df) else "direction_jitter"
    rank_colors = {
        1: "tab:red",
        2: "tab:orange",
        3: "gold",
        4: "tab:green",
        5: "tab:purple",
    }
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            score_df = camera_df[
                camera_df["grid_row"].astype(int).eq(gy)
                & camera_df["grid_col"].astype(int).eq(gx)
            ].sort_values("window_start_sec")
            if len(score_df):
                ax.plot(
                    score_df["window_start_sec"],
                    score_df["thresholded_direction_jitter_score"],
                    linewidth=1.1,
                    marker="o",
                    markersize=2.5,
                    color="tab:blue",
                    label="vibration score",
                    zorder=2,
                )
                topk_df = score_df[score_df.get("direction_jitter_topk_selected", pd.Series(False, index=score_df.index)).astype(bool)]
                topk_ranks = pd.to_numeric(topk_df["direction_jitter_topk_rank"], errors="coerce").fillna(0).astype(int)
                for rank, color in rank_colors.items():
                    rank_df = topk_df[topk_ranks.eq(rank)]
                    if rank_df.empty:
                        continue
                    ax.scatter(
                        rank_df["window_start_sec"],
                        rank_df["thresholded_direction_jitter_score"],
                        s=36,
                        color=color,
                        edgecolors="black",
                        linewidths=0.4,
                        label=f"top {rank}",
                        zorder=4,
                    )
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35, zorder=1)
            title = f"{gx + 1}x{gy + 1}_{camera}"
            if len(score_df):
                topk_count = int(score_df.get("direction_jitter_topk_selected", pd.Series(False, index=score_df.index)).astype(bool).sum())
                title += f" max={score_df['thresholded_direction_jitter_score'].max():.2f} top={topk_count}"
            ax.set_title(title, fontsize=10)
            ax.set_ylim(0.0, 1.05)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel("vibration score")
            if gy == grid_rows - 1:
                ax.set_xlabel("window start [sec]")
            handles, labels = ax.get_legend_handles_labels()
            if handles:
                by_label = dict(zip(labels, handles))
                ax.legend(by_label.values(), by_label.keys(), loc="best", fontsize=7)

    fig.suptitle(f"thresholded vibration score {camera} grid top{BROAD_VIBRATION_TOP_K}: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} direction jitter score rows found.")
    return axes


# 関数メモ: topk_mean
# - 動画単位集計で使う、上位K件の平均値を返します。
# - 単発ピークだけでなく、強い窓が複数ある動画を高く評価するための要約関数です。

def topk_mean(s: pd.Series, k: int = 5) -> float:
    return s.nlargest(min(k, len(s))).mean()


flow_window_score_tables = []
direction_jitter_score_tables = []
broad_vibration_score_tables = []
# raw flow 可視化を有効にした場合だけ、診断用の重いグラフをまとめて出力します。
if PLOT_FEATURE_GRAPHS and PLOT_RAW_FLOW_XY:
    for sample_id, result in target_results.items():
        raw_flow_df = result["raw_flow_df"]
        output_dir = Path(result["sample"]["output_dir"])
        plot_raw_flow_xy(raw_flow_df, "front", output_dir / f"{sample_id}_front_raw_flow_grid_xy_0p1s.png")
        plot_raw_flow_xy(raw_flow_df, "rear", output_dir / f"{sample_id}_rear_raw_flow_grid_xy_0p1s.png")

                # 可視化用に、front/rear それぞれのグリッド別振動スコア表を作ります。
        sample_direction_jitter_tables = []
        for camera in ["front", "rear"]:
            direction_score_df = build_flow_direction_jitter_score_table(raw_flow_df, camera)
            if len(direction_score_df):
                direction_score_df.insert(0, "video_id", sample_id)
                direction_score_df.insert(1, "target_label", result["sample"].get("plot_label", sample_id))
                direction_score_df.insert(2, "target_category", result["sample"].get("category", "unknown"))
                direction_score_df.insert(3, "target_environment", result["sample"].get("environment", "unknown"))
                sample_direction_jitter_tables.append(direction_score_df)
        sample_direction_jitter_df = pd.concat(sample_direction_jitter_tables, ignore_index=True) if sample_direction_jitter_tables else pd.DataFrame()
        if len(sample_direction_jitter_df):
            sample_direction_jitter_df = add_direction_jitter_topk_columns(sample_direction_jitter_df)
        result["direction_jitter_window_scores_df"] = sample_direction_jitter_df
        if len(sample_direction_jitter_df):
            sample_direction_jitter_path = output_dir / "direction_jitter_window_scores.csv"
            sample_direction_jitter_df.to_csv(sample_direction_jitter_path, index=False)
            direction_jitter_score_tables.append(sample_direction_jitter_df)
            print(f"saved: {sample_direction_jitter_path}")
                        # グリッド別振動スコアから、カメラ別の広域振動スコアを再集計して保存します。
            sample_broad_vibration_df = build_broad_vibration_score_table(sample_direction_jitter_df)
            result["broad_vibration_score_df"] = sample_broad_vibration_df
            sample_broad_vibration_path = output_dir / f"{sample_id}_broad_vibration_score_top{BROAD_VIBRATION_TOP_K}.csv"
            sample_broad_vibration_plot_path = output_dir / f"{sample_id}_broad_vibration_score_top{BROAD_VIBRATION_TOP_K}.png"
            sample_broad_vibration_df.to_csv(sample_broad_vibration_path, index=False)
            broad_vibration_score_tables.append(sample_broad_vibration_df)
            plot_broad_vibration_score(
                sample_broad_vibration_df,
                sample_broad_vibration_plot_path,
                title=f"broad vibration score by camera top{BROAD_VIBRATION_TOP_K}: {result['sample'].get('plot_label', sample_id)}",
            )
            print(f"saved: {sample_broad_vibration_path}")
            plot_direction_jitter_score_grid(
                sample_direction_jitter_df,
                "front",
                output_dir / f"{sample_id}_front_vibration_score_grid_top{BROAD_VIBRATION_TOP_K}.png",
            )
            plot_direction_jitter_score_grid(
                sample_direction_jitter_df,
                "rear",
                output_dir / f"{sample_id}_rear_vibration_score_grid_top{BROAD_VIBRATION_TOP_K}.png",
            )
        else:
            result["broad_vibration_score_df"] = pd.DataFrame()

        plot_raw_flow_direction_change(raw_flow_df, "front", output_dir / f"{sample_id}_front_raw_flow_grid_direction_change_0p1s.png", sample_direction_jitter_df)
        plot_raw_flow_direction_change(raw_flow_df, "rear", output_dir / f"{sample_id}_rear_raw_flow_grid_direction_change_0p1s.png", sample_direction_jitter_df)

        sample_window_score_tables = []
        for camera in ["front", "rear"]:
            for component in ["x", "y"]:
                window_score_df = build_flow_grid_window_score_table(raw_flow_df, camera, component)
                if len(window_score_df):
                    window_score_df.insert(0, "video_id", sample_id)
                    window_score_df.insert(1, "target_label", result["sample"].get("plot_label", sample_id))
                    window_score_df.insert(2, "target_category", result["sample"].get("category", "unknown"))
                    window_score_df.insert(3, "target_environment", result["sample"].get("environment", "unknown"))
                    sample_window_score_tables.append(window_score_df)
        sample_window_score_df = pd.concat(sample_window_score_tables, ignore_index=True) if sample_window_score_tables else pd.DataFrame()
        result["flow_window_scores_df"] = sample_window_score_df
        if len(sample_window_score_df):
            sample_window_score_path = output_dir / f"{sample_id}_flow_grid_window_scores.csv"
            sample_window_score_df.to_csv(sample_window_score_path, index=False)
            flow_window_score_tables.append(sample_window_score_df)
            print(f"saved: {sample_window_score_path}")

        plot_raw_flow_component_change_abs(raw_flow_df, "front", "x", output_dir / f"{sample_id}_front_raw_flow_grid_x_change_abs_0p1s.png", sample_window_score_df)
        plot_raw_flow_component_change_abs(raw_flow_df, "front", "y", output_dir / f"{sample_id}_front_raw_flow_grid_y_change_abs_0p1s.png", sample_window_score_df)
        plot_raw_flow_component_change_abs(raw_flow_df, "rear", "x", output_dir / f"{sample_id}_rear_raw_flow_grid_x_change_abs_0p1s.png", sample_window_score_df)
        plot_raw_flow_component_change_abs(raw_flow_df, "rear", "y", output_dir / f"{sample_id}_rear_raw_flow_grid_y_change_abs_0p1s.png", sample_window_score_df)
        plot_raw_flow_vector_change_abs(raw_flow_df, "front", output_dir / f"{sample_id}_front_raw_flow_grid_vector_change_abs_0p1s.png")
        plot_raw_flow_vector_change_abs(raw_flow_df, "rear", output_dir / f"{sample_id}_rear_raw_flow_grid_vector_change_abs_0p1s.png")
        plot_raw_flow_magnitude_change_abs(raw_flow_df, "front", output_dir / f"{sample_id}_front_raw_flow_grid_magnitude_change_abs_0p1s.png")
        plot_raw_flow_magnitude_change_abs(raw_flow_df, "rear", output_dir / f"{sample_id}_rear_raw_flow_grid_magnitude_change_abs_0p1s.png")
        plot_raw_flow_magnitude_change_abs_lowpass(raw_flow_df, "front", output_dir / f"{sample_id}_front_raw_flow_grid_magnitude_change_abs_lowpass_0p1s.png")
        plot_raw_flow_magnitude_change_abs_lowpass(raw_flow_df, "rear", output_dir / f"{sample_id}_rear_raw_flow_grid_magnitude_change_abs_lowpass_0p1s.png")
elif not PLOT_FEATURE_GRAPHS:
    print("Feature graphs skipped because PLOT_FEATURE_GRAPHS=False.")
elif not PLOT_RAW_FLOW_XY:
    print("Raw flow feature graphs skipped because PLOT_RAW_FLOW_XY=False.")

flow_grid_window_scores_df = pd.concat(flow_window_score_tables, ignore_index=True) if flow_window_score_tables else pd.DataFrame()
if len(flow_grid_window_scores_df):
    flow_window_score_path = TARGET_COMPARISON_DIR / "all_target_flow_grid_window_scores.csv"
    flow_grid_window_scores_df.to_csv(flow_window_score_path, index=False)
    display(flow_grid_window_scores_df.sort_values("flow_window_score", ascending=False).head(30))
    print(f"saved flow window scores: {flow_window_score_path}")
else:
    flow_grid_window_scores_df = pd.DataFrame()

direction_jitter_window_scores_df = pd.concat(direction_jitter_score_tables, ignore_index=True) if direction_jitter_score_tables else pd.DataFrame()
if len(direction_jitter_window_scores_df):
    direction_jitter_path = TARGET_COMPARISON_DIR / "all_target_direction_jitter_window_scores.csv"
    direction_jitter_window_scores_df.to_csv(direction_jitter_path, index=False)
    display(direction_jitter_window_scores_df.sort_values("direction_jitter_score", ascending=False).head(30))
    print(f"saved direction jitter window scores: {direction_jitter_path}")
else:
    direction_jitter_window_scores_df = pd.DataFrame()

broad_vibration_scores_df = pd.concat(broad_vibration_score_tables, ignore_index=True) if broad_vibration_score_tables else pd.DataFrame()
if len(broad_vibration_scores_df):
    broad_vibration_path = TARGET_COMPARISON_DIR / f"all_target_broad_vibration_score_top{BROAD_VIBRATION_TOP_K}.csv"
    broad_vibration_plot_path = TARGET_COMPARISON_DIR / f"overlay_broad_vibration_score_top{BROAD_VIBRATION_TOP_K}.png"
    broad_vibration_scores_df.to_csv(broad_vibration_path, index=False)
    plot_broad_vibration_score(
        broad_vibration_scores_df,
        broad_vibration_plot_path,
        title=f"broad vibration score by camera top{BROAD_VIBRATION_TOP_K} overlay",
    )
    display(broad_vibration_scores_df.sort_values("broad_vibration_score", ascending=False).head(30))
    print(f"saved broad vibration scores: {broad_vibration_path}")
    print(f"saved broad vibration plot  : {broad_vibration_plot_path}")
else:
    broad_vibration_scores_df = pd.DataFrame()

if RUN_INFERENCE and len(all_target_scored_df):
    summary_agg = {
        "max_anomaly_score": ("anomaly_score", "max"),
        "top5_mean_anomaly_score": ("anomaly_score", topk_mean),
        "p95_anomaly_score": ("anomaly_score", lambda s: s.quantile(0.95)),
        "n_windows": ("anomaly_score", "size"),
    }
    for col in ["audio_anomaly_score", "motion_anomaly_score", "sync_score"]:
        if col in all_target_scored_df.columns:
            summary_agg[f"max_{col}"] = (col, "max")
            summary_agg[f"top5_mean_{col}"] = (col, topk_mean)

    target_video_summary = (
        all_target_scored_df
        .groupby(["video_id", "target_category", "target_environment", "target_label"], as_index=False)
        .agg(**summary_agg)
        .sort_values("max_anomaly_score", ascending=False)
        .reset_index(drop=True)
    )

    # 依頼された確認用の推論結果表です。動画単位の最大スコアで並べます。
    result_source = target_video_summary.copy()
    for col in ["max_sync_score", "max_audio_anomaly_score", "max_motion_anomaly_score"]:
        if col not in result_source.columns:
            result_source[col] = 0.0
    inference_result_table = (
        result_source[[
            "video_id",
            "target_category",
            "target_environment",
            "max_anomaly_score",
            "max_sync_score",
            "max_audio_anomaly_score",
            "max_motion_anomaly_score",
        ]]
        .rename(columns={
            "video_id": "ファイル名",
            "target_category": "正解ラベル",
            "target_environment": "環境",
            "max_anomaly_score": "異常スコア",
            "max_sync_score": "同期異常スコア",
            "max_audio_anomaly_score": "音声異常スコア",
            "max_motion_anomaly_score": "動き異常スコア",
        })
        .sort_values("異常スコア", ascending=False)
        .reset_index(drop=True)
    )

    summary_path = TARGET_COMPARISON_DIR / "target_video_scores.csv"
    inference_result_table_path = TARGET_COMPARISON_DIR / "target_inference_result_table.csv"
    target_video_summary.to_csv(summary_path, index=False)
    inference_result_table.to_csv(inference_result_table_path, index=False)

    display(inference_result_table)

    score_cols = [c for c in PLOT_SCORE_COLUMNS if c in all_target_scored_df.columns]
    feature_cols = [c for c in PLOT_FEATURE_COLUMNS if PLOT_FEATURE_GRAPHS and c in all_target_scored_df.columns]
    top_cols = [
        "video_id", "target_category", "target_environment", "target_label", "time",
        *score_cols, "final_anomaly_score", "anomaly_score_smooth", *feature_cols,
    ]
    top_cols = list(dict.fromkeys([c for c in top_cols if c in all_target_scored_df.columns]))
    target_top_windows = pd.concat(
        [g.nlargest(TOP_N_ANOMALIES, "anomaly_score") for _, g in all_target_scored_df.groupby("video_id", sort=False)],
        ignore_index=True,
    )[top_cols]

    key_metric_cols = [
        "video_id", "target_category", "target_environment", "target_label", "time",
        *score_cols, "final_anomaly_score", "anomaly_score_smooth", *feature_cols,
    ]
    key_metric_cols = list(dict.fromkeys([c for c in key_metric_cols if c in all_target_scored_df.columns]))
    all_target_key_metrics_df = all_target_scored_df[key_metric_cols].copy()

    top_windows_path = TARGET_COMPARISON_DIR / "target_top_windows.csv"
    key_metrics_path = TARGET_COMPARISON_DIR / "all_target_key_metrics.csv"
    overlay_plot_path = TARGET_COMPARISON_DIR / "overlay_anomaly_scores.png"
    overlay_key_metrics_plot_path = TARGET_COMPARISON_DIR / "overlay_key_metrics.png"

    target_top_windows.to_csv(top_windows_path, index=False)
    all_target_key_metrics_df.to_csv(key_metrics_path, index=False)

    if PLOT_EXISTING_GRAPHS:
        plot_overlay_metrics(all_target_scored_df, score_cols, overlay_plot_path)
        if PLOT_FEATURE_GRAPHS and feature_cols:
            plot_overlay_metrics(all_target_scored_df, feature_cols, overlay_key_metrics_plot_path)

    for sample_id, result in target_results.items():
        output_dir = Path(result["sample"]["output_dir"])
        scored_df = result["scored_df"]
        target_video_summary[target_video_summary["video_id"].astype(str).eq(str(sample_id))].to_csv(output_dir / f"{sample_id}_video_score.csv", index=False)
        scored_df.nlargest(TOP_N_ANOMALIES, "anomaly_score").to_csv(output_dir / f"{sample_id}_top_windows.csv", index=False)
        if PLOT_EXISTING_GRAPHS:
            plot_anomaly_scores(scored_df, output_dir / f"{sample_id}_anomaly_plot.png")

    display(target_video_summary)
    display(target_top_windows)
    display(all_target_key_metrics_df.head(12))
    print(f"saved summary              : {summary_path}")
    print(f"saved inference result table: {inference_result_table_path}")
    print(f"saved top windows          : {top_windows_path}")
    print(f"saved key metrics          : {key_metrics_path}")
    if PLOT_EXISTING_GRAPHS:
        print(f"saved score overlay        : {overlay_plot_path}")
        if PLOT_FEATURE_GRAPHS and feature_cols:
            print(f"saved feature overlay      : {overlay_key_metrics_plot_path}")
        elif not PLOT_FEATURE_GRAPHS:
            print("Feature overlay skipped because PLOT_FEATURE_GRAPHS=False.")
else:
    target_video_summary = pd.DataFrame()
    inference_result_table = pd.DataFrame()
    target_top_windows = pd.DataFrame()
    all_target_key_metrics_df = pd.DataFrame()
    if not RUN_INFERENCE:
        print("Inference result table skipped because RUN_INFERENCE=False.")
    else:
        print("Inference result table skipped because no scored windows were produced.")


Feature graphs skipped because PLOT_FEATURE_GRAPHS=False.


,ファイル名,正解ラベル,環境,異常スコア,同期異常スコア,音声異常スコア,動き異常スコア
0,023_旋回時、柱に衝突,anomaly,indoor,1.000000,1.000000,1.000000,1.000000
1,043_ラック下を通過時マストが衝突,anomaly,indoor,1.000000,1.000000,1.000000,1.000000
2,086_前進時、ラックに衝突,anomaly,indoor,1.000000,1.000000,1.000000,1.000000
3,001_後進時にトラックに衝突,anomaly,outdoor,0.988509,0.987483,1.000000,0.975123
4,054_荷が崩落,anomaly,indoor,0.980189,0.981856,0.964041,1.000000
5,073_旋回時、ラックに衝突,anomaly,indoor,0.959644,0.962745,0.926877,1.000000
6,010_人身ヒヤリ,anomaly,indoor,0.959241,0.962367,1.000000,1.000000
7,1012,normal,outdoor,0.950413,0.943669,1.000000,0.927948
8,088_前進時、筋交いに追突,anomaly,indoor,0.928780,0.933442,0.871315,1.000000
9,030_旋回時、安全ポール衝突＆後方から作業者,anomaly,indoor,0.928089,0.943722,0.936977,1.000000


,video_id,target_category,target_environment,target_label,max_anomaly_score,top5_mean_anomaly_score,p95_anomaly_score,n_windows,max_audio_anomaly_score,top5_mean_audio_anomaly_score,max_motion_anomaly_score,top5_mean_motion_anomaly_score,max_sync_score,top5_mean_sync_score
0,023_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/023,1.000000,0.979912,0.943959,74,1.000000,0.955359,1.000000,1.000000,1.000000,1.000000
1,043_ラック下を通過時マストが衝突,anomaly,indoor,anomaly/indoor/043,1.000000,0.941005,0.873998,73,1.000000,0.899944,1.000000,1.000000,1.000000,1.000000
2,086_前進時、ラックに衝突,anomaly,indoor,anomaly/indoor/086,1.000000,0.935903,0.853525,73,1.000000,0.894702,1.000000,1.000000,1.000000,1.000000
3,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,0.988509,0.900110,0.830198,73,1.000000,0.902373,0.975123,0.974802,0.987483,0.987321
4,054_荷が崩落,anomaly,indoor,anomaly/indoor/054,0.980189,0.884086,0.779823,77,0.964041,0.852865,1.000000,1.000000,0.981856,0.970929
5,073_旋回時、ラックに衝突,anomaly,indoor,anomaly/indoor/073,0.959644,0.792241,0.664274,75,0.926877,0.810098,1.000000,1.000000,0.962745,0.906775
6,010_人身ヒヤリ,anomaly,indoor,anomaly/indoor/010,0.959241,0.803832,0.730078,75,1.000000,0.872882,1.000000,1.000000,0.962367,0.962367
7,1012,normal,outdoor,normal/outdoor/1012,0.950413,0.881760,0.810689,72,1.000000,0.896036,0.927948,0.912974,0.943669,0.920888
8,088_前進時、筋交いに追突,anomaly,indoor,anomaly/indoor/088,0.928780,0.838834,0.672919,74,0.871315,0.833334,1.000000,0.913105,0.933442,0.883373
9,030_旋回時、安全ポール衝突＆後方から作業者,anomaly,indoor,anomaly/indoor/030,0.928089,0.783323,0.707565,74,0.936977,0.888033,1.000000,0.992735,0.943722,0.935986


,video_id,target_category,target_environment,target_label,time,anomaly_score,audio_anomaly_score,motion_anomaly_score,sync_score,final_anomaly_score,anomaly_score_smooth
0,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,9.4,0.904841,0.828450,1.000000,0.910192,0.904841,0.754952
1,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,9.6,0.867224,0.751889,0.992828,0.906922,0.867224,0.740879
2,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,9.8,0.844694,0.701823,0.992828,0.906922,0.844694,0.700454
3,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,9.2,0.582702,0.112585,1.000000,0.910192,0.582702,0.667889
4,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,9.0,0.575301,0.096139,1.000000,0.910192,0.575301,0.572107
...,...,...,...,...,...,...,...,...,...,...,...
395,1086,normal,indoor,normal/indoor/1086,5.4,0.589111,0.380042,0.866612,0.573890,0.589111,0.501568
396,1086,normal,indoor,normal/indoor/1086,5.0,0.551337,0.283729,0.866612,0.601725,0.551337,0.500040
397,1086,normal,indoor,normal/indoor/1086,5.2,0.547945,0.276191,0.866612,0.601725,0.547945,0.518045
398,1086,normal,indoor,normal/indoor/1086,12.6,0.546234,0.136116,0.930648,0.796273,0.546234,0.513903


,video_id,target_category,target_environment,target_label,time,anomaly_score,audio_anomaly_score,motion_anomaly_score,sync_score,final_anomaly_score,anomaly_score_smooth
0,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,0.0,0.121295,0.083907,0.118671,0.210008,0.121295,0.223271
1,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,0.2,0.219733,0.273469,0.118671,0.275683,0.219733,0.233515
2,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,0.4,0.328787,0.371646,0.277914,0.321381,0.328787,0.239630
3,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,0.6,0.264247,0.228224,0.277914,0.321381,0.264247,0.266511
4,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,0.8,0.264090,0.227874,0.277914,0.321381,0.264090,0.277912
5,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,1.0,0.255698,0.260369,0.242092,0.268998,0.255698,0.266278
6,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,1.2,0.276741,0.298644,0.242092,0.288093,0.276741,0.245848
7,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,1.4,0.270616,0.293571,0.242092,0.268885,0.270616,0.233371
8,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,1.6,0.162096,0.171367,0.126413,0.203683,0.162096,0.216177
9,084_旋回時、柱に衝突,anomaly,indoor,anomaly/indoor/084,1.8,0.201706,0.242278,0.126413,0.242185,0.201706,0.179058


saved summary              : /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/target_video_scores.csv
saved inference result table: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/target_inference_result_table.csv
saved top windows          : /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/target_top_windows.csv
saved key metrics          : /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/all_target_key_metrics.csv


## 6. ピーク窓の簡易確認

In [8]:
# ------------------------------------------------------------
# セル概要: 上位窓の寄与特徴確認
# ------------------------------------------------------------
# - ピーク窓について、モデル入力空間で値が大きい特徴を簡易的に並べます。
# - IsolationForest の厳密な説明ではなく、どの特徴が目立っていたかを見るためのラフな確認です。
# ------------------------------------------------------------

# ============================================================
# スコアが高かった窓の特徴量確認
# ------------------------------------------------------------
# IsolationForest の厳密な寄与度ではありませんが、標準化後の入力値が
# 大きい特徴を並べることで、どの特徴が強く出ていたかを確認します。
# ============================================================
# 関数メモ: top_contributing_features
# - 指定した窓について、モデル入力空間で絶対値が大きい特徴を上位表示します。
# - IsolationForest の完全な説明ではありませんが、どの特徴が目立っていたかを簡易確認できます。

def top_contributing_features(model_name: str, row_index: int, model_inputs: dict[str, np.ndarray], top_n: int = 10) -> pd.DataFrame:
    artifact = score_model_artifacts[model_name]
    values = model_inputs[model_name][row_index]
    feature_names = np.asarray(artifact["feature_names"])
    order = np.argsort(np.abs(values))[::-1][:top_n]
    return pd.DataFrame({
        "model_name": model_name,
        "feature": feature_names[order],
        "group": [artifact.get("expanded_feature_group_map", {}).get(str(feature_names[i]), "other") for i in order],
        "scaled_value": values[order],
        "abs_scaled_value": np.abs(values[order]),
    })


if RUN_INFERENCE:
    for sample_id, result in target_results.items():
        scored_df = result["scored_df"]
        if scored_df is None or scored_df.empty:
            continue
        peak_idx = int(scored_df["anomaly_score"].idxmax())
        peak_pos = scored_df.index.get_loc(peak_idx)
        print("=" * 80)
        show_cols = ["video_id", "target_environment", "time", *[c for c in PLOT_SCORE_COLUMNS if c in scored_df.columns], "final_anomaly_score", "anomaly_score_smooth"]
        show_cols = list(dict.fromkeys(show_cols))
        print(scored_df.loc[peak_idx, show_cols])
        for model_name in score_model_artifacts:
            display(top_contributing_features(model_name, peak_pos, result["model_inputs"], top_n=12))
else:
    print("Top contributing features skipped because RUN_INFERENCE=False.")


video_id                084_旋回時、柱に衝突
target_environment            indoor
time                             9.4
anomaly_score               0.904841
audio_anomaly_score          0.82845
motion_anomaly_score             1.0
sync_score                  0.910192
final_anomaly_score         0.904841
anomaly_score_smooth        0.754952
Name: 47, dtype: object


,model_name,feature,group,scaled_value,abs_scaled_value
0,audio,audio_mel_15,audio_mel,3.240742,3.240742
1,audio,audio_mel_14,audio_mel,2.928354,2.928354
2,audio,audio_bandwidth,audio_basic,2.851745,2.851745
3,audio,audio_mel_13,audio_mel,2.789611,2.789611
4,audio,audio_mel_12,audio_mel,2.561951,2.561951
5,audio,audio_mel_05,audio_mel,2.218900,2.218900
6,audio,audio_mel_11,audio_mel,2.076370,2.076370
7,audio,audio_mel_08,audio_mel,2.043660,2.043660
8,audio,audio_mel_04,audio_mel,2.014936,2.014936
9,audio,audio_mel_07,audio_mel,1.811308,1.811308


,model_name,feature,group,scaled_value,abs_scaled_value
0,motion,front_broad_vibration_score,broad_vibration,3.129923,3.129923
1,motion,rear_broad_vibration_score,broad_vibration,-0.234476,0.234476


video_id                    1030
target_environment        indoor
time                        11.6
anomaly_score           0.723061
audio_anomaly_score     0.515912
motion_anomaly_score    0.993475
sync_score              0.715923
final_anomaly_score     0.723061
anomaly_score_smooth    0.541164
Name: 58, dtype: object


,model_name,feature,group,scaled_value,abs_scaled_value
0,audio,audio_mel_15,audio_mel,2.311836,2.311836
1,audio,audio_mel_14,audio_mel,2.095622,2.095622
2,audio,audio_mel_13,audio_mel,1.864087,1.864087
3,audio,audio_mel_00,audio_mel,1.723626,1.723626
4,audio,audio_centroid,audio_basic,-1.418853,1.418853
5,audio,audio_mel_12,audio_mel,1.407510,1.407510
6,audio,audio_mel_05,audio_mel,1.266671,1.266671
7,audio,audio_mel_01,audio_mel,1.244360,1.244360
8,audio,audio_mel_06,audio_mel,1.092694,1.092694
9,audio,audio_mel_02,audio_mel,1.086092,1.086092


...

,model_name,feature,group,scaled_value,abs_scaled_value
0,audio,audio_mel_01,audio_mel,-2.999794,2.999794
1,audio,audio_mel_00,audio_mel,-2.983047,2.983047
2,audio,audio_mel_06,audio_mel,-2.941750,2.941750
3,audio,audio_mel_02,audio_mel,-2.906073,2.906073
4,audio,audio_mel_07,audio_mel,-2.892309,2.892309
5,audio,audio_mel_08,audio_mel,-2.711230,2.711230
6,audio,audio_mel_03,audio_mel,-2.654641,2.654641
7,audio,audio_mel_05,audio_mel,-2.644053,2.644053
8,audio,audio_mel_04,audio_mel,-2.486311,2.486311
9,audio,audio_mel_09,audio_mel,-2.299781,2.299781


,model_name,feature,group,scaled_value,abs_scaled_value
0,motion,rear_broad_vibration_score,broad_vibration,2.649003,2.649003
1,motion,front_broad_vibration_score,broad_vibration,-0.249927,0.249927


video_id                    1086
target_environment        indoor
time                        12.2
anomaly_score           0.767285
audio_anomaly_score     0.681301
motion_anomaly_score     0.86127
sync_score              0.796273
final_anomaly_score     0.767285
anomaly_score_smooth    0.581409
Name: 61, dtype: object


,model_name,feature,group,scaled_value,abs_scaled_value
0,audio,audio_mel_06,audio_mel,-2.078193,2.078193
1,audio,audio_mel_02,audio_mel,-1.912527,1.912527
2,audio,audio_mel_07,audio_mel,-1.894498,1.894498
3,audio,audio_mel_05,audio_mel,-1.819744,1.819744
4,audio,audio_mel_03,audio_mel,-1.808270,1.808270
5,audio,audio_mel_01,audio_mel,-1.758663,1.758663
6,audio,audio_mel_04,audio_mel,-1.639733,1.639733
7,audio,audio_mel_00,audio_mel,-1.634931,1.634931
8,audio,audio_zcr,audio_basic,1.287152,1.287152
9,audio,audio_mel_08,audio_mel,-1.251726,1.251726


,model_name,feature,group,scaled_value,abs_scaled_value
0,motion,rear_broad_vibration_score,broad_vibration,1.958879,1.958879
1,motion,front_broad_vibration_score,broad_vibration,-0.933565,0.933565
